# KLDM CASAL Velocity Impulse Tests

This notebook follows the updated velocity-impulse theory: residual -> velocity impulse -> KLDM/TDM coordinate move -> CASCAL z/mu update -> tangent velocity projection for the next step.


In [1]:
import dataclasses
import math
import os
import random
import sys
import time
import traceback
from collections import Counter
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Optional

import numpy as np
import pandas as pd
import torch
import yaml
from IPython.display import display

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / 'configs').exists():
    ROOT = ROOT.parent
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from kldmPlus.algorithm10_casal_chart import Algorithm10Config, _decode_lattice_matrix, _encode_lattice_matrix, _k_to_l, _l_to_k
from kldmPlus.algorithm13_fixed_template_velocity_casal import (
    FixedTemplateVelocityConfig,
    apply_full_space_force,
    apply_reduced_space_force,
    build_cartesian_block_metric,
    build_fractional_metric,
    center_velocity,
    compute_template_jacobian,
    finite_difference_jacobian_error,
    graph_velocity_norm,
    kinetic_position_update,
    lift_reduced_chart_velocity,
    materialize_template,
    mean_norm,
    projector_idempotence_error,
    project_to_fixed_template,
    reduced_chart_velocity,
    site_jacobian_block_ranks,
    tangent_project_velocity,
    tangent_projector,
    tangent_residual_after_centering,
    wrap_residual,
)
from kldmPlus.algorithm14_kldm_casal_velocity_impulse import (
    CASAL_THEOREM_APPLIES,
    CASAL_THEOREM_REASON,
    VelocityImpulseConfig,
    build_relaxed_target,
    center_velocity as center_velocity_impulse,
    kldm_casal_velocity_impulse_step,
    kinetic_position_update_signed,
    tangent_project_mean_free,
    update_dual,
    velocity_score_prefactor,
    velocity_score_variance,
    wrapdiff,
)
from kldmPlus.run_sampling_compare import SamplingCompareRunner, set_seed
from kldmPlus.sample_evaluation import evaluate_csp_reconstruction
from kldmPlus.symmetry.frame_bridge import map_standardized_structure_to_vanilla_frame, standardize_structure
from kldmPlus.symmetry import (
    materialize_pcs_state,
    sample_random_free_vars,
    select_requested_template_state,
    select_requested_template_states,
    validate_requested_space_group,
)
from kldmPlus.symmetry.pcs_projection import (
    _build_vanilla_structure,
    _collapse_centering_equivalent_structure,
    _periodic_pairwise_distances,
    refresh_pcs_state_anchor,
    vanilla_structure_to_model_tensors,
)
from kldmPlus.utils.time import iter_sampling_times

CONFIG_PATH = ROOT / 'configs' / 'kldm_plus' / 'mp_20' / 'mp20_sampling_compare_em_vs_algorithm10_casal_chart.yaml'
with CONFIG_PATH.open('r', encoding='utf-8') as handle:
    CONFIG = yaml.safe_load(handle)
COMPARE_CFG = CONFIG['sampling_compare']

PROFILE = os.environ.get('FIXED_TEMPLATE_VELOCITY_PROFILE', 'laptop').strip().lower()

def profile_default(name: str, laptop: str, deep: str | None = None) -> str:
    if name in os.environ:
        return os.environ[name]
    if PROFILE in {'full', 'deep'} and deep is not None:
        return deep
    return laptop


def parse_int_list(text: str) -> list[int]:
    return [int(v.strip()) for v in str(text).split(',') if v.strip()]


def parse_float_list(text: str) -> list[float]:
    return [float(v.strip()) for v in str(text).split(',') if v.strip()]

SAMPLE_SEED = int(profile_default('FIXED_TEMPLATE_VELOCITY_SEED', '7', '7'))
GRAPH_IDS = parse_int_list(profile_default('FIXED_TEMPLATE_VELOCITY_GRAPH_IDS', '1,2,3', '1,2,3'))
CAPTURE_N_STEPS = int(profile_default('FIXED_TEMPLATE_VELOCITY_CAPTURE_N_STEPS', '80', '160'))
CAPTURE_STEP = int(profile_default('FIXED_TEMPLATE_VELOCITY_CAPTURE_STEP', '72', '144'))
CAPTURE_SWEEP_STEPS = parse_int_list(profile_default('FIXED_TEMPLATE_VELOCITY_CAPTURE_SWEEP', '16,32,48,64,80', '32,64,96,128,160'))
MINI_CHAIN_STEPS = int(profile_default('FIXED_TEMPLATE_VELOCITY_MINI_CHAIN_STEPS', '12', '20'))
FULL_CHAIN_STEPS = int(profile_default('FIXED_TEMPLATE_VELOCITY_FULL_CHAIN_STEPS', '40', '80'))
LATE_START_STEP = int(profile_default('FIXED_TEMPLATE_VELOCITY_LATE_START_STEP', '32', '64'))
TEMPLATE_TOP_K = int(profile_default('FIXED_TEMPLATE_VELOCITY_TEMPLATE_TOP_K', '3', '5'))
FIXTURE_MAX_TEMPLATES = int(profile_default('FIXED_TEMPLATE_VELOCITY_MAX_TEMPLATES', '12', '20'))
FIXTURE_TEMPLATE_EVAL_LIMIT = int(profile_default('FIXED_TEMPLATE_VELOCITY_TEMPLATE_EVAL_LIMIT', '4', '6'))
FIXTURE_OPT_STEPS = int(profile_default('FIXED_TEMPLATE_VELOCITY_OPT_STEPS', '80', '120'))
FIXTURE_LR = float(profile_default('FIXED_TEMPLATE_VELOCITY_LR', '5e-2', '5e-2'))
ORACLE_W_MODE = str(profile_default('FIXED_TEMPLATE_VELOCITY_ORACLE_W_MODE', '1', '1')).strip().lower() not in {'0', 'false', 'no'}
EPS_PASS = 1.0e-6
FD_EPS_VALUES = parse_float_list(profile_default('FIXED_TEMPLATE_VELOCITY_FD_EPS', '1e-2,1e-3,1e-4', '1e-2,1e-3,1e-4'))
METRIC_MODES = [part.strip() for part in profile_default('FIXED_TEMPLATE_VELOCITY_METRICS', 'fractional,cartesian', 'fractional,cartesian').split(',') if part.strip()]
print(f'profile={PROFILE} graphs={GRAPH_IDS} capture_step={CAPTURE_STEP}/{CAPTURE_N_STEPS} mini_chain_steps={MINI_CHAIN_STEPS} full_chain_steps={FULL_CHAIN_STEPS} oracle_W_mode={ORACLE_W_MODE}')

KAPPA_SWEEP = parse_float_list(profile_default('KLDM_CASAL_IMPULSE_KAPPA_SWEEP', '0.1,0.3,1.0,3.0,10.0', '0.1,0.3,1.0,3.0,10.0'))
RHO_SWEEP = parse_float_list(profile_default('KLDM_CASAL_IMPULSE_RHO_SWEEP', '0.1,1.0,10.0', '0.1,1.0,10.0'))
DUAL_MODES = [part.strip() for part in profile_default('KLDM_CASAL_IMPULSE_DUAL_MODES', 'none,tau_over_rho,tau_times_rho', 'none,tau_over_rho,tau_times_rho').split(',') if part.strip()]
TANGENT_MODES = [part.strip() for part in profile_default('KLDM_CASAL_IMPULSE_TANGENT_MODES', 'none,after_move_combined', 'none,after_move_combined').split(',') if part.strip()]
BETA_SWEEP = parse_float_list(profile_default('KLDM_CASAL_IMPULSE_BETA_SWEEP', '1e-4,3e-4,1e-3,3e-3,1e-2', '1e-4,3e-4,1e-3,3e-3,1e-2'))


MODELS_PROJECT_ROOT: /workspace/.venv/lib/python3.11/site-packages/mattergen
profile=laptop graphs=[1, 2, 3] capture_step=72/80 mini_chain_steps=12 full_chain_steps=40 oracle_W_mode=True


In [2]:
set_seed(SAMPLE_SEED)
runner = SamplingCompareRunner(config_path=CONFIG_PATH)
runner.model.eval()
template_prior = runner._ensure_template_prior()
batch = next(iter(runner.loader)).to(runner.device)
ptr = batch.ptr.tolist()

sampler_cfgs = {cfg['name']: dict(cfg) for cfg in COMPARE_CFG['samplers']}
facit_cfg = sampler_cfgs.get('FacitKLDM_PC', COMPARE_CFG['samplers'][0])
algo10_cfg_map = dict(sampler_cfgs['Algorithm10_CASAL_chart'].get('algorithm10', {}))
BASE_ALGO10 = Algorithm10Config.from_mapping(algo10_cfg_map)


def replace_supported_dataclass(obj, **updates):
    supported = {field.name for field in dataclasses.fields(obj)}
    return dataclasses.replace(obj, **{key: value for key, value in updates.items() if key in supported})


BASE_GUIDE_ALGO10 = replace_supported_dataclass(
    BASE_ALGO10,
    debug=False,
    oracle_wyckoff_debug=True,
    return_best_even_if_invalid=True,
    max_templates=FIXTURE_MAX_TEMPLATES,
    template_eval_limit=FIXTURE_TEMPLATE_EVAL_LIMIT,
    ssvd_max_steps=min(int(getattr(BASE_ALGO10, 'ssvd_max_steps', 8)), 6),
    ssvd_random_restarts=0,
)

FIXED_CFG = FixedTemplateVelocityConfig(projector_damping=1.0e-6)
IMPULSE_BASE_CFG = VelocityImpulseConfig(sign_chi=-1.0, projector_damping=1.0e-6, kappa_coord=1.0, projection=FIXED_CFG, relaxed_target_mode='cascal', dual_mode='tau_over_rho', tangent_mode='after_move_combined')

@dataclass
class GraphCase:
    graph_id: int
    graph_idx0: int
    composition: dict[int, int]
    atomic_numbers: torch.Tensor
    gt_frac: torch.Tensor
    gt_lattice: torch.Tensor
    requested_sg: int


@dataclass
class KLDMState:
    f: torch.Tensor
    v: torch.Tensor
    l: torch.Tensor
    k: torch.Tensor
    h: torch.Tensor
    t: float
    dt: float
    graph_idx0: int
    full_state: Optional[dict[str, Any]] = None
    full_times: Optional[Any] = None


@dataclass
class FixedTemplateFixture:
    case: GraphCase
    state: Any
    theta: torch.Tensor
    tau: torch.Tensor
    z_frac: torch.Tensor
    z_k: torch.Tensor
    z_l: torch.Tensor
    assignment: torch.Tensor
    J: torch.Tensor
    template_labels: str
    template_multiplicities: str
    template_dofs: str
    template_total_dof: int
    requested_sg_match: bool
    composition_match: bool
    projection_objective: float
    projection_rank: int | None
    projection_condition: float | None
    config: FixedTemplateVelocityConfig
    chart_frac: torch.Tensor | None = None
    chart_atomic_numbers: torch.Tensor | None = None
    chart_l: torch.Tensor | None = None
    chart_k: torch.Tensor | None = None
    chart_num_atoms: int = 0
    raw_num_atoms: int = 0
    target_representation_name: str = ''
    anchor_representation_name: str = ''
    uses_oracle_chart: bool = False
    oracle_w_payload: dict[str, Any] | None = None


result_tables: dict[str, pd.DataFrame] = {}
_caches: dict[tuple[Any, ...], Any] = {}


def graph_slice(graph_idx0: int) -> tuple[int, int]:
    return int(ptr[graph_idx0]), int(ptr[graph_idx0 + 1])


def composition_counter(values) -> dict[int, int]:
    arr = [int(v) for v in torch.as_tensor(values, dtype=torch.long).reshape(-1).tolist()]
    return dict(sorted(Counter(arr).items()))


def graph_tensors(graph_idx0: int, *, source=None) -> dict[str, Any]:
    start, end = graph_slice(graph_idx0)
    if source is None:
        pos, l, h = batch.pos, batch.l, batch.atomic_numbers
    else:
        if len(source) == 4:
            pos, _v, l, h = source
        else:
            pos, l, h = source
    return {
        'pos': pos[start:end].detach().clone(),
        'l': l[graph_idx0].detach().clone().reshape(-1),
        'h': h[start:end].detach().clone().to(torch.long),
        'sg': int(torch.as_tensor(batch.space_group).reshape(-1)[graph_idx0].item()),
        'num_atoms': int(end - start),
    }


def load_test_graphs(graph_ids=GRAPH_IDS) -> list[GraphCase]:
    out = []
    for graph_id in graph_ids:
        graph_idx0 = int(graph_id) - 1
        g = graph_tensors(graph_idx0)
        out.append(
            GraphCase(
                graph_id=int(graph_id),
                graph_idx0=graph_idx0,
                composition=composition_counter(g['h']),
                atomic_numbers=g['h'],
                gt_frac=g['pos'],
                gt_lattice=g['l'],
                requested_sg=g['sg'],
            )
        )
    return out


GRAPH_CASES = load_test_graphs()
backend_summary = {
    'state_backend': 'KLDM reverse sampler via SamplingCompareRunner/model._prepare_csp_sampling + reverse_exp_step',
    'projection_backend': 'fixed_template_ssvd_project via algorithm14_kldm_casal_velocity_impulse (faithful velocity impulse + CASCAL z/mu)',
    'oracle_w_source': 'oracle structure -> select_requested_template_states -> PCSTemplateState/template extractor -> Phi_W/J_W',
}
print(f'loaded_graphs={[g.graph_id for g in GRAPH_CASES]} requested_sg={[g.requested_sg for g in GRAPH_CASES]}')
print('backend_summary=', backend_summary)

print(f'CASAL_THEOREM_APPLIES={CASAL_THEOREM_APPLIES} REASON={CASAL_THEOREM_REASON}')


mps:  False
dataset_cache action=load dataset=mp_20 split=test reason=fresh path=/workspace/data/mp_20/processed/test
dataset_cache action=from_cache_path:start dataset=mp_20 split=test
dataset_cache action=from_cache_path:done dataset=mp_20 split=test
dataset_cache action=builder_build:start dataset=mp_20 split=test
dataset_cache action=builder_build:done dataset=mp_20 split=test
dataset_cache action=load dataset=mp_20 split=train reason=fresh path=/workspace/data/mp_20/processed/train
dataset_cache action=from_cache_path:start dataset=mp_20 split=train
dataset_cache action=from_cache_path:done dataset=mp_20 split=train
dataset_cache action=builder_build:start dataset=mp_20 split=train
dataset_cache action=builder_build:done dataset=mp_20 split=train
template_prior_cache_hit path=/workspace/notebooks/.cache/kldmPlus/template_prior/0448db66e575a7bb.pkl records=43
loaded_graphs=[1, 2, 3] requested_sg=[227, 4, 194]
backend_summary= {'state_backend': 'KLDM reverse sampler via SamplingComp

In [3]:
def dataframe_result(name: str, rows: list[dict[str, Any]]) -> pd.DataFrame:
    df = pd.DataFrame(rows)
    if 'PASS' not in df.columns:
        df['PASS'] = False
    if 'status' not in df.columns:
        df['status'] = np.where(df['PASS'], 'PASS', 'FAIL')
    result_tables[name] = df
    return df


def error_row(test: str, graph: int | str, exc: Exception, failure_category: str, **extra) -> dict[str, Any]:
    tb = traceback.format_exc().strip().splitlines()
    return {
        'test': test,
        'graph': graph,
        'PASS': False,
        'status': 'ERROR',
        'error_type': type(exc).__name__,
        'error_message': str(exc),
        'traceback_tail': tb[-1] if tb else '',
        'failure_category': failure_category,
        **extra,
    }


def print_group_pass(df: pd.DataFrame, group_cols) -> None:
    if df.empty:
        print('empty')
        return
    if isinstance(group_cols, str):
        group_cols = [group_cols]
    cols = list(group_cols) + ['PASS', 'status']
    display(df[cols].groupby(group_cols, as_index=False).agg({'PASS': 'all', 'status': 'last'}))


def l_to_k(l: torch.Tensor, graph_case: GraphCase) -> torch.Tensor:
    return _l_to_k(
        l=l.reshape(-1),
        num_atoms=int(graph_case.gt_frac.shape[0]),
        lattice_transform=runner.lattice_transform,
    ).to(device=l.device, dtype=l.dtype)


def k_to_l(k: torch.Tensor, graph_case: GraphCase) -> torch.Tensor:
    return _k_to_l(
        k=k.reshape(-1),
        num_atoms=int(graph_case.gt_frac.shape[0]),
        lattice_transform=runner.lattice_transform,
    ).to(device=k.device, dtype=k.dtype)


def ensure_l_feature(l: torch.Tensor, num_atoms: int) -> torch.Tensor:
    flat = l.reshape(-1)
    if int(flat.numel()) == 9:
        encoded = _encode_lattice_matrix(
            cell_matrix=flat.reshape(3, 3),
            num_atoms=int(num_atoms),
            lattice_transform=runner.lattice_transform,
        )
        return encoded.to(device=l.device, dtype=l.dtype).reshape(-1)
    return flat


def cell_from_l(l: torch.Tensor, num_atoms: int) -> torch.Tensor:
    flat = l.reshape(-1)
    if int(flat.numel()) == 9:
        return flat.reshape(3, 3).detach()
    return _decode_lattice_matrix(
        l=flat,
        num_atoms=int(num_atoms),
        lattice_transform=runner.lattice_transform,
    ).detach()


def volume_from_l(l: torch.Tensor, num_atoms: int) -> float:
    return float(torch.abs(torch.linalg.det(cell_from_l(l, num_atoms))).detach().item())


def min_pair_distance_from_arrays(frac: torch.Tensor, l: torch.Tensor, num_atoms: int) -> float:
    cell = cell_from_l(l, num_atoms).to(device=frac.device, dtype=frac.dtype)
    distances = _periodic_pairwise_distances(frac_coords=frac, cell_matrix=cell)
    return float(distances.min().detach().item()) if distances.numel() else float('inf')


def cart_distance_to_projection(frac: torch.Tensor, projection, num_atoms: int) -> float:
    cell = cell_from_l(projection.z_l, num_atoms).to(device=frac.device, dtype=frac.dtype)
    delta = wrap_residual(frac, projection.z_frac) @ cell
    return float(torch.linalg.norm(delta.reshape(-1)).detach().item())


def clone_full_state(full_state: dict[str, Any]) -> dict[str, Any]:
    cloned: dict[str, Any] = {}
    for key, value in full_state.items():
        cloned[key] = value.clone() if torch.is_tensor(value) else value
    return cloned


def _native_reverse_step_full_state(full_state: dict[str, Any], times) -> dict[str, Any]:
    with torch.no_grad():
        preds_curr = full_state['score_network'](
            t=times.now.graph,
            pos=full_state['f_t'],
            v=full_state['v_t'],
            h=full_state['a_t'],
            l=full_state['l_t'],
            node_index=full_state['node_index'],
            edge_node_index=full_state['edge_node_index'],
        )
        score_v = full_state['sampling_tdm'].reconstruct_full_reverse_velocity_score(
            t=times.now.nodes,
            v_t=full_state['v_t'],
            pred_v=preds_curr['v'],
            index=full_state['node_index'],
        )
        full_state['f_t'], full_state['v_t'] = full_state['sampling_tdm'].reverse_exp_step(
            f_t=full_state['f_t'],
            v_t=full_state['v_t'],
            score_v=score_v,
            index=full_state['node_index'],
            dt=times.dt,
        )
        full_state['l_t'] = runner.model._reverse_lattice_sampling_step(
            t=times.now.lattice,
            x_t=full_state['l_t'],
            pred=preds_curr['l'],
            dt=times.dt,
            num_atoms=batch.num_atoms,
        )
    return full_state


def graph_state_from_full(full_state: dict[str, Any], case: GraphCase, times=None) -> KLDMState:
    start, end = graph_slice(case.graph_idx0)
    f = full_state['f_t'][start:end].detach().clone()
    v = full_state['v_t'][start:end].detach().clone()
    l = full_state['l_t'][case.graph_idx0].detach().clone().reshape(-1)
    h = full_state['a_t'][start:end].detach().clone().to(torch.long)
    k = l_to_k(l, case)
    dt = float(times.dt) if times is not None else 1.0 / max(1, CAPTURE_N_STEPS)
    t = float(times.now.graph[case.graph_idx0].detach().reshape(-1)[0].item()) if times is not None else float('nan')
    return KLDMState(f=f, v=v, l=l, k=k, h=h, t=t, dt=dt, graph_idx0=case.graph_idx0, full_state=full_state, full_times=times)


def capture_batch_kldm_state(seed=SAMPLE_SEED, n_steps=CAPTURE_N_STEPS, capture_step=CAPTURE_STEP):
    key = ('capture', int(seed), int(n_steps), int(capture_step))
    if key in _caches:
        return _caches[key]
    set_seed(seed)
    state = runner.model._prepare_csp_sampling(
        batch=batch,
        n_steps=int(n_steps),
        t_start=float(facit_cfg.get('t_start', 1.0)),
        t_final=float(facit_cfg.get('t_final', 1e-3)),
    )
    last_times = None
    for step_idx, times in enumerate(iter_sampling_times(batch=state['batch'], grid=state['sampling_time_grid']), start=1):
        last_times = times
        state = _native_reverse_step_full_state(state, times)
        if step_idx >= int(capture_step):
            break
    _caches[key] = (state, last_times, int(capture_step))
    return _caches[key]

# Canonical state-lifting, evaluation, and score-compatibility helpers live in Cell 5.


In [4]:


def metric_tensor(metric_name: str, l: torch.Tensor, num_atoms: int) -> torch.Tensor | None:
    metric_name = str(metric_name).strip().lower()
    if metric_name == 'cartesian':
        cell = cell_from_l(l, num_atoms).to(device=l.device, dtype=l.dtype)
        return build_cartesian_block_metric(cell, num_atoms=num_atoms)
    return build_fractional_metric(num_atoms=num_atoms, dtype=l.dtype, device=l.device)


def template_labels(template) -> str:
    return ','.join(f"{site.atomic_number}@{site.label}" for site in template.site_templates)


def template_multiplicities(template) -> str:
    return ','.join(str(int(site.multiplicity)) for site in template.site_templates)


def template_dofs(template) -> str:
    return ','.join(str(int(site.dof)) for site in template.site_templates)


def l_to_k_num_atoms(l: torch.Tensor, num_atoms: int) -> torch.Tensor:
    return _l_to_k(
        l=ensure_l_feature(l, int(num_atoms)).reshape(-1),
        num_atoms=int(num_atoms),
        lattice_transform=runner.lattice_transform,
    ).to(device=l.device, dtype=l.dtype)


def k_to_l_num_atoms(k: torch.Tensor, num_atoms: int) -> torch.Tensor:
    return _k_to_l(
        k=k.reshape(-1),
        num_atoms=int(num_atoms),
        lattice_transform=runner.lattice_transform,
    ).to(device=k.device, dtype=k.dtype)


def _pcs_state_chart_target(state0) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor, str, str]:
    chart_frac = state0.target_frac if state0.target_frac is not None else state0.anchor_frac
    chart_atomic_numbers = state0.target_atomic_numbers if state0.target_atomic_numbers is not None else state0.anchor_atomic_numbers
    chart_cell = state0.target_cell if state0.target_cell is not None else state0.anchor_cell
    chart_k = state0.target_k if state0.target_k is not None else state0.anchor_k
    if chart_frac is None or chart_atomic_numbers is None or chart_cell is None or chart_k is None:
        raise RuntimeError('PCS state is missing oracle chart tensors.')
    return (
        chart_frac.detach().clone(),
        chart_atomic_numbers.detach().clone().to(torch.long),
        chart_cell.detach().clone(),
        chart_k.detach().clone(),
        str(state0.target_representation_name or 'standardized'),
        str(state0.anchor_representation_name or 'standardized'),
    )


def _oracle_w_payload_from_state(case: GraphCase, state0, *, chart_num_atoms: int, chart_repr: str, anchor_repr: str) -> dict[str, Any]:
    return {
        'space_group': int(case.requested_sg),
        'template_labels': template_labels(state0.template),
        'template_multiplicities': template_multiplicities(state0.template),
        'template_dofs': template_dofs(state0.template),
        'template_total_dof': int(state0.template.total_free_dims),
        'template_num_atoms': int(state0.template.total_atoms),
        'chart_num_atoms': int(chart_num_atoms),
        'raw_num_atoms': int(case.gt_frac.shape[0]),
        'target_representation': chart_repr,
        'anchor_representation': anchor_repr,
        'species_assignment_mode': 'oracle_fixed_template',
        'origin_gauge_mode': 'pcs_anchor',
        'extraction_backend': 'select_requested_template_states -> PCSTemplateState',
        'jacobian_backend': 'compute_template_jacobian(materialize_template)',
        'projection_backend': 'fixed_template_ssvd_project',
        'state_backend': 'KLDM reverse sampler',
    }


def _chart_structure_to_vanilla_tensors(
    case: GraphCase,
    fixture: FixedTemplateFixture,
    *,
    frac_coords: torch.Tensor,
    atomic_numbers: torch.Tensor,
    l: torch.Tensor,
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    cell = cell_from_l(l, int(frac_coords.shape[0])).to(device=frac_coords.device, dtype=frac_coords.dtype)
    structure = _build_vanilla_structure(
        frac_coords=frac_coords,
        atomic_numbers=atomic_numbers,
        cell_matrix=cell,
    )
    standardized_like = structure
    translations = fixture.state.target_centering_translations
    if translations is not None and int(atomic_numbers.shape[0]) != int(case.atomic_numbers.shape[0]):
        try:
            standardized_like = _collapse_centering_equivalent_structure(
                structure=structure,
                translations=translations,
                expected_atomic_numbers=fixture.state.bridge.vanilla_atomic_numbers,
            )
        except Exception:
            standardized_like = structure
    try:
        vanilla = map_standardized_structure_to_vanilla_frame(
            standardized_structure=standardized_like,
            vanilla_reference_structure=fixture.state.bridge.vanilla_structure,
            symprec=fixture.state.bridge.symprec,
            angle_tolerance=fixture.state.bridge.angle_tolerance,
        )
    except Exception:
        vanilla = standardized_like
    return vanilla_structure_to_model_tensors(
        structure=vanilla,
        lattice_transform=runner.lattice_transform,
        device=frac_coords.device,
        dtype=frac_coords.dtype,
    )


def _validate_projection_for_case(case: GraphCase, projection, fixture: FixedTemplateFixture | None = None) -> Any:
    structure = _build_vanilla_structure(
        frac_coords=projection.z_frac,
        atomic_numbers=projection.raw.atomic_numbers_chart,
        cell_matrix=projection.raw.cell,
    )
    expected_atomic_numbers = case.atomic_numbers
    if fixture is not None and ORACLE_W_MODE and fixture.uses_oracle_chart:
        expected_atomic_numbers = fixture.chart_atomic_numbers
    return validate_requested_space_group(
        structure=structure,
        requested_space_group=case.requested_sg,
        expected_atomic_numbers=expected_atomic_numbers,
    )


def build_fixed_template_fixture(case: GraphCase) -> FixedTemplateFixture:
    target_cell = cell_from_l(case.gt_lattice, int(case.gt_frac.shape[0])).to(device=case.gt_frac.device, dtype=case.gt_frac.dtype)
    oracle_structure = _build_vanilla_structure(
        frac_coords=case.gt_frac,
        atomic_numbers=case.atomic_numbers,
        cell_matrix=target_cell,
    )

    def _select_candidates(standardization: str):
        return select_requested_template_states(
            frac_coords=case.gt_frac,
            atomic_numbers=case.atomic_numbers,
            cell_matrix=target_cell,
            space_group_number=case.requested_sg,
            standardization=standardization,
            symprec=1e-2,
            angle_tolerance=5.0,
            max_templates=max(FIXTURE_MAX_TEMPLATES, 16),
            template_eval_limit=max(FIXTURE_TEMPLATE_EVAL_LIMIT, 6),
            optimization_steps=FIXTURE_OPT_STEPS,
            learning_rate=FIXTURE_LR,
            coord_weight=1.0,
            lattice_weight=0.0,
            pairdist_weight=0.0,
            steric_weight=0.0,
            volume_weight=0.0,
            k6_weight=0.0,
            freeze_lattice_free_vars=True,
            quick_templates=False,
            top_k=max(TEMPLATE_TOP_K, 6),
            template_prior=template_prior,
            template_prior_weight=0.0,
            oracle_reference_structure=oracle_structure,
            oracle_fit_structure=oracle_structure,
        )

    candidate_entries = []
    selection_errors = []
    for standardization in ['conventional', 'primitive']:
        try:
            states = _select_candidates(standardization)
        except Exception as exc:
            selection_errors.append((standardization, exc))
            continue
        for state0 in states:
            chart_frac, chart_atomic_numbers, chart_cell, chart_k, target_repr, anchor_repr = _pcs_state_chart_target(state0)
            candidate_entries.append({
                'standardization': standardization,
                'state': state0,
                'chart_frac': chart_frac,
                'chart_atomic_numbers': chart_atomic_numbers,
                'chart_cell': chart_cell,
                'chart_k': chart_k,
                'chart_num_atoms': int(chart_atomic_numbers.shape[0]),
                'target_repr': target_repr,
                'anchor_repr': anchor_repr,
                'same_count': int(chart_atomic_numbers.shape[0]) == int(case.gt_frac.shape[0]),
                'expanded': ('expanded' in target_repr) or ('expanded' in anchor_repr) or int(chart_atomic_numbers.shape[0]) != int(case.gt_frac.shape[0]),
            })

    if not candidate_entries:
        lines = [f"{std}:{type(exc).__name__}:{exc}" for std, exc in selection_errors]
        raise RuntimeError(
            f'No fixed-template candidates could be selected for graph={case.graph_id}. selection_errors={lines}'
        )

    candidate_entries.sort(
        key=lambda item: (
            0 if item['standardization'] == 'conventional' else 1,
            0 if item['same_count'] else 1,
            float(item['state'].objective),
        )
    )

    candidate_cfgs = [
        FIXED_CFG,
        dataclasses.replace(
            FIXED_CFG,
            projection_config=dataclasses.replace(FIXED_CFG.projection_config, use_fixed_assignment=False),
        ),
    ]
    best_payload = None
    errors = []
    for item in candidate_entries:
        tau0 = torch.zeros(1, 3, device=item['chart_frac'].device, dtype=item['chart_frac'].dtype)
        for cfg0 in candidate_cfgs:
            try:
                projection = project_to_fixed_template(
                    f_frac=item['chart_frac'],
                    atomic_numbers=item['chart_atomic_numbers'],
                    template_state=item['state'],
                    target_k=item['chart_k'],
                    tau0=tau0,
                    config=cfg0,
                )
                J = compute_template_jacobian(projection.theta, projection.raw.state, tau=projection.tau)
                fixture = FixedTemplateFixture(
                    case=case,
                    state=projection.raw.state,
                    theta=projection.theta,
                    tau=projection.tau,
                    z_frac=projection.z_frac,
                    z_k=projection.z_k,
                    z_l=projection.z_l,
                    assignment=projection.assignment,
                    J=J,
                    template_labels=template_labels(projection.raw.state.template),
                    template_multiplicities=template_multiplicities(projection.raw.state.template),
                    template_dofs=template_dofs(projection.raw.state.template),
                    template_total_dof=int(projection.raw.state.template.total_free_dims),
                    requested_sg_match=False,
                    composition_match=False,
                    projection_objective=float(projection.objective),
                    projection_rank=int(getattr(projection.raw, 'ssvd_rank', 0)) if getattr(projection.raw, 'ssvd_rank', None) is not None else None,
                    projection_condition=float(getattr(projection.raw, 'ssvd_condition_number', float('nan'))) if getattr(projection.raw, 'ssvd_condition_number', None) is not None else None,
                    config=cfg0,
                    chart_frac=item['chart_frac'].detach().clone(),
                    chart_atomic_numbers=item['chart_atomic_numbers'].detach().clone(),
                    chart_l=ensure_l_feature(item['chart_cell'], item['chart_num_atoms']),
                    chart_k=item['chart_k'].detach().clone(),
                    chart_num_atoms=item['chart_num_atoms'],
                    raw_num_atoms=int(case.gt_frac.shape[0]),
                    target_representation_name=item['target_repr'],
                    anchor_representation_name=item['anchor_repr'],
                    uses_oracle_chart=bool(ORACLE_W_MODE),
                    oracle_w_payload=_oracle_w_payload_from_state(
                        case,
                        projection.raw.state,
                        chart_num_atoms=item['chart_num_atoms'],
                        chart_repr=item['target_repr'],
                        anchor_repr=item['anchor_repr'],
                    ),
                )
                validation = _validate_projection_for_case(case, projection, fixture)
                coord_residual = float(torch.linalg.norm(wrap_residual(item['chart_frac'], projection.z_frac).reshape(-1)).detach().item())
                score = (
                    0 if validation.requested_space_group_match else 1,
                    0 if validation.composition_match else 1,
                    0 if item['standardization'] == 'conventional' else 1,
                    0 if bool(getattr(cfg0.projection_config, 'use_fixed_assignment', False)) else 1,
                    coord_residual,
                    float(projection.objective),
                )
                if best_payload is None or score < best_payload[0]:
                    best_payload = (score, fixture, validation)
            except Exception as exc:
                errors.append(
                    f"std={item['standardization']} chart_atoms={item['chart_num_atoms']} target_repr={item['target_repr']} "
                    f"fixed={bool(getattr(cfg0.projection_config, 'use_fixed_assignment', False))} {type(exc).__name__}:{exc}"
                )

    if best_payload is None:
        candidate_summary = [
            {
                'std': item['standardization'],
                'chart_atoms': item['chart_num_atoms'],
                'target_repr': item['target_repr'],
                'anchor_repr': item['anchor_repr'],
                'template_labels': template_labels(item['state'].template),
            }
            for item in candidate_entries[:8]
        ]
        raise RuntimeError(
            f'Could not build an oracle-W fixed template for graph={case.graph_id}. '
            f'candidate_summary={candidate_summary} errors={errors[:8]}'
        )

    _, fixture, validation = best_payload
    fixture.requested_sg_match = bool(validation.requested_space_group_match)
    fixture.composition_match = bool(validation.composition_match)
    return fixture


def _state_to_fixture_chart(case: GraphCase, fixture: FixedTemplateFixture, state: KLDMState) -> KLDMState:
    if not ORACLE_W_MODE or not fixture.uses_oracle_chart:
        return state
    if int(state.f.shape[0]) == int(fixture.chart_num_atoms) and int(state.h.shape[0]) == int(fixture.chart_num_atoms):
        return state
    cell = cell_from_l(state.l, int(state.f.shape[0])).to(device=state.f.device, dtype=state.f.dtype)
    refreshed_state = refresh_pcs_state_anchor(
        state=fixture.state,
        frac_coords=torch.remainder(state.f, 1.0),
        atomic_numbers=state.h.to(device=state.f.device, dtype=torch.long),
        cell_matrix=cell,
        pairdist_weight=0.0,
        pairdist_bins=32,
        pairdist_max_distance=6.0,
        pairdist_bandwidth=0.15,
        coord_weight=1.0,
        lattice_weight=0.0,
        optimization_steps=max(8, min(int(FIXTURE_OPT_STEPS), 32)),
        learning_rate=float(FIXTURE_LR),
    )
    chart_frac, chart_atomic_numbers, chart_cell, chart_k, target_repr, anchor_repr = _pcs_state_chart_target(refreshed_state)
    if int(chart_atomic_numbers.shape[0]) != int(fixture.chart_num_atoms):
        raise RuntimeError(
            f'oracle chart lift mismatch graph={case.graph_id} expected_chart_atoms={fixture.chart_num_atoms} got={int(chart_atomic_numbers.shape[0])} '
            f'target_repr={target_repr!r} anchor_repr={anchor_repr!r} template_atoms={int(refreshed_state.template.total_atoms)}'
        )
    chart_l = ensure_l_feature(chart_cell, int(chart_frac.shape[0]))
    if int(chart_frac.shape[0]) == int(state.v.shape[0]):
        chart_v = state.v.detach().clone()
    else:
        multiplier = max(1, int(chart_frac.shape[0]) // max(int(state.v.shape[0]), 1))
        chart_v = state.v.repeat(multiplier, 1)
    return KLDMState(
        f=chart_frac.detach().clone(),
        v=chart_v.detach().clone(),
        l=chart_l.detach().clone(),
        k=chart_k.detach().clone(),
        h=chart_atomic_numbers.detach().clone().to(device=state.f.device, dtype=torch.long),
        t=state.t,
        dt=state.dt,
        graph_idx0=state.graph_idx0,
        full_state=state.full_state,
        full_times=state.full_times,
    )

def _maybe_lift_state_to_fixture_chart(case: GraphCase, state: KLDMState) -> KLDMState:
    fixture = globals().get('fixtures', {}).get(case.graph_id)
    if fixture is None:
        return state
    return _state_to_fixture_chart(case, fixture, state)


def gt_state_for_graph(case: GraphCase) -> KLDMState:
    f = case.gt_frac.detach().clone()
    l = case.gt_lattice.detach().clone().reshape(-1)
    h = case.atomic_numbers.detach().clone().to(torch.long)
    k = l_to_k(l, case)
    v = torch.zeros_like(f)
    return _maybe_lift_state_to_fixture_chart(case, KLDMState(f=f, v=v, l=l, k=k, h=h, t=0.0, dt=0.0, graph_idx0=case.graph_idx0))


def random_fractional_state(case: GraphCase, *, seed_offset: int = 0) -> KLDMState:
    generator = torch.Generator(device=case.gt_frac.device).manual_seed(SAMPLE_SEED + int(case.graph_id) + int(seed_offset))
    f = torch.rand(case.gt_frac.shape, generator=generator, device=case.gt_frac.device, dtype=case.gt_frac.dtype)
    l = case.gt_lattice.detach().clone().reshape(-1)
    h = case.atomic_numbers.detach().clone().to(torch.long)
    k = l_to_k(l, case)
    v = torch.zeros_like(f)
    return _maybe_lift_state_to_fixture_chart(case, KLDMState(f=f, v=v, l=l, k=k, h=h, t=float('nan'), dt=1.0 / max(1, CAPTURE_N_STEPS), graph_idx0=case.graph_idx0))


def sample_kldm_state_for_graph_at_step(case: GraphCase, *, capture_step=CAPTURE_STEP, n_steps=CAPTURE_N_STEPS, seed=SAMPLE_SEED) -> KLDMState:
    full_state, times, _step_idx = capture_batch_kldm_state(seed=seed, n_steps=n_steps, capture_step=capture_step)
    return _maybe_lift_state_to_fixture_chart(case, graph_state_from_full(clone_full_state(full_state), case, times=times))


def evaluate_arrays(case: GraphCase, pred_f: torch.Tensor, pred_l: torch.Tensor, pred_h: torch.Tensor) -> dict[str, Any]:
    fixture = globals().get('fixtures', {}).get(case.graph_id)
    eval_f = pred_f
    eval_l = pred_l
    eval_h = pred_h.to(torch.long)
    if fixture is not None and ORACLE_W_MODE and fixture.uses_oracle_chart and int(eval_h.shape[0]) != int(case.atomic_numbers.shape[0]):
        eval_f, eval_l, eval_h = _chart_structure_to_vanilla_tensors(
            case,
            fixture,
            frac_coords=pred_f,
            atomic_numbers=eval_h,
            l=pred_l,
        )
    pred_l_feature = ensure_l_feature(eval_l, int(eval_f.shape[0]))
    result = evaluate_csp_reconstruction(
        pred_f=eval_f,
        pred_l=pred_l_feature,
        pred_a=eval_h,
        target_f=case.gt_frac,
        target_l=case.gt_lattice,
        target_a=case.atomic_numbers,
        lattice_transform=runner.lattice_transform,
        requested_space_group=case.requested_sg,
        validity_cutoff=float(getattr(BASE_ALGO10, 'hard_min_distance', 0.5)),
    )
    standardized_frac_rmse = None
    if result.matcher_diagnostics is not None:
        standardized_frac_rmse = result.matcher_diagnostics.standardized_frac_rmse
    return {
        'match': bool(result.match),
        'valid': bool(result.valid),
        'rmse': result.rmse,
        'frac_rmse': result.frac_rmse,
        'standardized_frac_rmse': standardized_frac_rmse,
        'composition_match': result.composition_match,
        'requested_sg_match': result.requested_space_group_match,
        'min_pair_distance': result.min_pair_distance,
        'volume': result.volume,
        'lattice_lengths_mae': result.lattice_lengths_mae,
        'lattice_angles_mae': result.lattice_angles_mae,
        'validity_reason': result.validity_reason,
    }


def score_norms_from_modified_state(case: GraphCase, base_state: KLDMState, *, f: torch.Tensor, v: torch.Tensor, l: torch.Tensor, h: torch.Tensor | None = None) -> dict[str, Any]:
    if base_state.full_state is None or base_state.full_times is None:
        return {
            'pred_v_norm': float('nan'),
            'score_v_norm': float('nan'),
            'pred_l_norm': float('nan'),
            'finite_ok': False,
        }
    if int(f.shape[0]) != int(case.gt_frac.shape[0]):
        return {
            'pred_v_norm': float('nan'),
            'score_v_norm': float('nan'),
            'pred_l_norm': float('nan'),
            'finite_ok': False,
        }
    full_state = clone_full_state(base_state.full_state)
    times = base_state.full_times
    start, end = graph_slice(case.graph_idx0)
    full_state['f_t'][start:end] = f.to(device=full_state['f_t'].device, dtype=full_state['f_t'].dtype)
    full_state['v_t'][start:end] = v.to(device=full_state['v_t'].device, dtype=full_state['v_t'].dtype)
    if h is not None and int(torch.as_tensor(h).reshape(-1).numel()) == int(end - start):
        full_state['a_t'][start:end] = torch.as_tensor(h, device=full_state['a_t'].device, dtype=full_state['a_t'].dtype).reshape(end-start)
    l_feature = ensure_l_feature(l, int(case.gt_frac.shape[0]))
    full_state['l_t'][case.graph_idx0] = l_feature.to(device=full_state['l_t'].device, dtype=full_state['l_t'].dtype)
    with torch.no_grad():
        preds = full_state['score_network'](
            t=times.now.graph,
            pos=full_state['f_t'],
            v=full_state['v_t'],
            h=full_state['a_t'],
            l=full_state['l_t'],
            node_index=full_state['node_index'],
            edge_node_index=full_state['edge_node_index'],
        )
        score_v = full_state['sampling_tdm'].reconstruct_full_reverse_velocity_score(
            t=times.now.nodes,
            v_t=full_state['v_t'],
            pred_v=preds['v'],
            index=full_state['node_index'],
        )
    pred_v_graph = preds['v'][start:end]
    score_v_graph = score_v[start:end]
    pred_l_graph = preds['l'][case.graph_idx0].reshape(-1)
    finite_ok = bool(torch.isfinite(pred_v_graph).all() and torch.isfinite(score_v_graph).all() and torch.isfinite(pred_l_graph).all())
    return {
        'pred_v_norm': float(torch.linalg.norm(pred_v_graph.reshape(-1)).detach().item()),
        'score_v_norm': float(torch.linalg.norm(score_v_graph.reshape(-1)).detach().item()),
        'pred_l_norm': float(torch.linalg.norm(pred_l_graph).detach().item()),
        'finite_ok': finite_ok,
    }


def project_state_to_fixture(
    case: GraphCase,
    fixture: FixedTemplateFixture,
    f_frac: torch.Tensor,
    k: torch.Tensor | None = None,
    *,
    use_self_target: bool = False,
    atomic_numbers: torch.Tensor | None = None,
    l_feature: torch.Tensor | None = None,
):
    if atomic_numbers is None:
        if ORACLE_W_MODE and fixture.uses_oracle_chart and int(f_frac.shape[0]) == int(fixture.chart_num_atoms):
            target_atomic_numbers = fixture.chart_atomic_numbers.detach().clone().to(device=f_frac.device, dtype=torch.long)
        else:
            target_atomic_numbers = case.atomic_numbers.detach().clone().to(device=f_frac.device, dtype=torch.long)
    else:
        target_atomic_numbers = atomic_numbers.detach().clone().to(device=f_frac.device, dtype=torch.long)
    if l_feature is None:
        if ORACLE_W_MODE and fixture.uses_oracle_chart and int(f_frac.shape[0]) == int(fixture.chart_num_atoms):
            base_l = fixture.chart_l.detach().clone()
        elif k is not None:
            base_l = k_to_l_num_atoms(k, int(f_frac.shape[0]))
        else:
            base_l = case.gt_lattice
    else:
        base_l = l_feature
    state_like = KLDMState(
        f=f_frac.detach().clone(),
        v=torch.zeros_like(f_frac),
        l=base_l.detach().clone(),
        k=(fixture.chart_k if k is None else k).detach().clone(),
        h=target_atomic_numbers.detach().clone(),
        t=float('nan'),
        dt=0.0,
        graph_idx0=case.graph_idx0,
    )
    if int(state_like.f.shape[0]) != int(fixture.chart_num_atoms) or int(state_like.h.shape[0]) != int(fixture.chart_num_atoms):
        state_like = _state_to_fixture_chart(case, fixture, state_like)
    target_k = (fixture.chart_k if k is None else state_like.k).detach().clone().to(device=state_like.f.device, dtype=state_like.f.dtype)
    template_state = fixture.state
    if use_self_target:
        target_frac = torch.remainder(state_like.f.detach().clone(), 1.0)
        target_l = state_like.l.detach().clone()
        target_cell = cell_from_l(target_l, int(target_frac.shape[0])).to(device=state_like.f.device, dtype=state_like.f.dtype)
        identity = torch.arange(int(target_frac.shape[0]), device=state_like.f.device, dtype=torch.long)
        template_state = dataclasses.replace(
            template_state,
            target_frac=target_frac,
            target_atomic_numbers=state_like.h.detach().clone(),
            target_cell=target_cell.detach().clone(),
            target_k=target_k.detach().clone(),
            fixed_target_assignment=identity.detach().clone(),
            anchor_frac=target_frac.detach().clone(),
            anchor_atomic_numbers=state_like.h.detach().clone(),
            anchor_cell=target_cell.detach().clone(),
            anchor_k=target_k.detach().clone(),
            anchor_assignment=identity.detach().clone(),
        )
    projection = project_to_fixed_template(
        f_frac=state_like.f,
        atomic_numbers=state_like.h,
        template_state=template_state,
        target_k=target_k,
        tau0=fixture.tau,
        config=fixture.config,
    )
    J = compute_template_jacobian(projection.theta, projection.raw.state, tau=projection.tau)
    return projection, J


def template_distance(
    case: GraphCase,
    fixture: FixedTemplateFixture,
    f_frac: torch.Tensor,
    k: torch.Tensor | None = None,
    *,
    use_self_target: bool = False,
    atomic_numbers: torch.Tensor | None = None,
    l_feature: torch.Tensor | None = None,
):
    projection, J = project_state_to_fixture(
        case,
        fixture,
        f_frac,
        k=k,
        use_self_target=use_self_target,
        atomic_numbers=atomic_numbers,
        l_feature=l_feature,
    )
    target_frac = projection.raw.state.target_frac if projection.raw.state.target_frac is not None else fixture.chart_frac
    dist = float(torch.linalg.norm(wrap_residual(target_frac, projection.z_frac).reshape(-1)).detach().item())
    return dist, projection, J


def one_step_repair(case: GraphCase, fixture: FixedTemplateFixture, state: KLDMState, *, mode: str, metric_name: str = 'fractional', eta: float = 0.05):
    state = _state_to_fixture_chart(case, fixture, state)
    projection, J = project_state_to_fixture(case, fixture, state.f, k=state.k, atomic_numbers=state.h, l_feature=state.l)
    metric = metric_tensor(metric_name, projection.z_l.to(device=state.f.device, dtype=state.f.dtype), int(state.f.shape[0]))
    residual = wrap_residual(state.f, projection.z_frac)
    tangent_velocity, projector, mean_norm_before = tangent_project_velocity(
        state.v,
        J=J,
        metric=metric,
        damping=FIXED_CFG.projector_damping,
        mean_free=FIXED_CFG.mean_free_velocity,
    )
    dist_after_projection, _projection_anchor, _J_anchor = template_distance(
        case,
        fixture,
        projection.z_frac,
        k=projection.z_k,
        use_self_target=True,
        atomic_numbers=projection.raw.atomic_numbers_chart,
        l_feature=projection.z_l,
    )
    if mode == 'none':
        f_new = state.f.detach().clone()
        v_new = state.v.detach().clone()
    elif mode == 'final_projection_only':
        f_new = projection.z_frac.detach().clone()
        v_new = state.v.detach().clone()
    elif mode == 'velocity_tangent_only':
        v_new = tangent_velocity
        f_new = kinetic_position_update(state.f, v_new, state.dt)
    elif mode == 'coord_projection_plus_tangent':
        v_new = tangent_velocity
        f_new = kinetic_position_update(projection.z_frac, v_new, state.dt)
    elif mode == 'normal_force_plus_tangent':
        forced = apply_full_space_force(state.v, residual=-residual, step_size=eta, mean_free=False)
        v_new, _projector2, _ = tangent_project_velocity(forced, J=J, metric=metric, damping=FIXED_CFG.projector_damping, mean_free=True)
        f_new = kinetic_position_update(state.f, v_new, state.dt)
    elif mode == 'reduced_space':
        omega = reduced_chart_velocity(state.v, J=J, metric=metric, damping=FIXED_CFG.projector_damping)
        residual_theta = reduced_chart_velocity(residual, J=J, metric=metric, damping=FIXED_CFG.projector_damping)
        omega = apply_reduced_space_force(omega, residual=-residual_theta, step_size=eta)
        v_new = lift_reduced_chart_velocity(omega, J=J, shape=state.v.shape)
        v_new, mean_norm_before = center_velocity(v_new)
        f_new = kinetic_position_update(state.f, v_new, state.dt)
    else:
        raise ValueError(f'unsupported repair mode: {mode}')
    repaired = KLDMState(
        f=f_new,
        v=v_new,
        l=state.l.detach().clone(),
        k=state.k.detach().clone(),
        h=state.h.detach().clone(),
        t=state.t,
        dt=state.dt,
        graph_idx0=state.graph_idx0,
        full_state=state.full_state,
        full_times=state.full_times,
    )
    dist_final, final_projection, final_J = template_distance(case, fixture, repaired.f, k=repaired.k, atomic_numbers=repaired.h, l_feature=repaired.l)
    info = {
        'dist_before': float(torch.linalg.norm(residual.reshape(-1)).detach().item()),
        'dist_after_projection': dist_after_projection,
        'dist_after': dist_final,
        'velocity_norm_before': graph_velocity_norm(state.v),
        'velocity_norm_after': graph_velocity_norm(v_new),
        'mean_norm_before': mean_norm_before,
        'mean_norm_after': mean_norm(v_new),
        'tangent_residual': projector.residual_norm(v_new),
        'projection_objective': float(projection.objective),
        'projection_energy_final': float(final_projection.objective),
        'rank_J': int(projector.rank),
        'cond_J': float(projector.condition_number),
        'min_pair_distance': min_pair_distance_from_arrays(repaired.f, repaired.l, int(repaired.f.shape[0])),
        'volume_ratio': volume_from_l(repaired.l, int(repaired.f.shape[0])) / max(volume_from_l(fixture.chart_l, int(fixture.chart_num_atoms)), 1.0e-8),
    }
    return repaired, projection, J, final_projection, final_J, info


def __run_short_chain_base(case: GraphCase, fixture: FixedTemplateFixture, *, mode: str, start_step: int, end_step: int, metric_name: str = 'fractional', eta: float = 0.05, final_projection: bool = False):
    set_seed(SAMPLE_SEED)
    full_state = runner.model._prepare_csp_sampling(
        batch=batch,
        n_steps=CAPTURE_N_STEPS,
        t_start=float(facit_cfg.get('t_start', 1.0)),
        t_final=float(facit_cfg.get('t_final', 1e-3)),
    )
    dists = []
    projection_fail_count = 0
    start_time = float('nan')
    end_time = float('nan')
    for step_idx, times in enumerate(iter_sampling_times(batch=full_state['batch'], grid=full_state['sampling_time_grid']), start=1):
        if step_idx < start_step:
            full_state = _native_reverse_step_full_state(full_state, times)
            continue
        if step_idx > end_step:
            break
        if math.isnan(start_time):
            start_time = float(times.now.graph[case.graph_idx0].detach().reshape(-1)[0].item())
        full_state = _native_reverse_step_full_state(full_state, times)
        state = graph_state_from_full(full_state, case, times)
        if mode != 'none':
            try:
                repaired, _proj, _J, _proj_final, _J_final, _info = one_step_repair(case, fixture, state, mode=mode, metric_name=metric_name, eta=eta)
                start, end = graph_slice(case.graph_idx0)
                if int(repaired.f.shape[0]) == int(end - start):
                    full_state['f_t'][start:end] = repaired.f.to(device=full_state['f_t'].device, dtype=full_state['f_t'].dtype)
                    full_state['v_t'][start:end] = repaired.v.to(device=full_state['v_t'].device, dtype=full_state['v_t'].dtype)
            except Exception:
                projection_fail_count += 1
        state = graph_state_from_full(full_state, case, times)
        dist_now, _proj_now, _J_now = template_distance(case, fixture, state.f, k=state.k, atomic_numbers=state.h, l_feature=state.l)
        dists.append(dist_now)
        end_time = float(times.now.graph[case.graph_idx0].detach().reshape(-1)[0].item())
    final_state = graph_state_from_full(full_state, case, times)
    if final_projection:
        proj_end, _J_end = project_state_to_fixture(case, fixture, final_state.f, k=final_state.k, atomic_numbers=final_state.h, l_feature=final_state.l)
        final_state = KLDMState(
            f=proj_end.z_frac.detach().clone(),
            v=final_state.v.detach().clone(),
            l=final_state.l.detach().clone(),
            k=proj_end.z_k.detach().clone(),
            h=proj_end.raw.atomic_numbers_chart.detach().clone(),
            t=final_state.t,
            dt=final_state.dt,
            graph_idx0=final_state.graph_idx0,
            full_state=final_state.full_state,
            full_times=final_state.full_times,
        )
        end_dist, _proj_end2, _J_end2 = template_distance(case, fixture, final_state.f, k=final_state.k, atomic_numbers=final_state.h, l_feature=final_state.l)
        dists.append(end_dist)
    evaluation = evaluate_arrays(case, final_state.f, final_state.l, final_state.h)
    score_norms = score_norms_from_modified_state(case, final_state, f=final_state.f, v=final_state.v, l=final_state.l)
    return {
        'dist_initial': dists[0] if dists else float('nan'),
        'dist_final': dists[-1] if dists else float('nan'),
        'projection_fail_count': int(projection_fail_count),
        'start_time': start_time,
        'end_time': end_time,
        'velocity_norm_max': float(max([0.0] + [graph_velocity_norm(final_state.v)])),
        'score_norm_max': float(max(score_norms['score_v_norm'], score_norms['pred_l_norm'])) if score_norms['finite_ok'] else float('nan'),
        **evaluation,
    }


def run_short_chain(case: GraphCase, fixture: FixedTemplateFixture, *, mode: str, start_step: int, end_step: int, metric_name: str = 'fractional', eta: float = 0.05, final_projection: bool = False):
    if ORACLE_W_MODE and fixture.uses_oracle_chart and fixture.chart_num_atoms != fixture.raw_num_atoms:
        dists = []
        projection_fail_count = 0
        start_time = float('nan')
        end_time = float('nan')
        final_state = None
        for step_idx in range(int(start_step), int(end_step) + 1):
            raw_state = sample_kldm_state_for_graph_at_step(case, capture_step=step_idx, n_steps=CAPTURE_N_STEPS)
            state = _state_to_fixture_chart(case, fixture, raw_state)
            if math.isnan(start_time):
                start_time = float(state.t)
            if mode != 'none':
                try:
                    repaired, _proj, _J, _proj_final, _J_final, _info = one_step_repair(case, fixture, state, mode=mode, metric_name=metric_name, eta=eta)
                    state = repaired
                except Exception:
                    projection_fail_count += 1
            dist_now, _proj_now, _J_now = template_distance(case, fixture, state.f, k=state.k, atomic_numbers=state.h, l_feature=state.l)
            dists.append(dist_now)
            end_time = float(state.t)
            final_state = state
        if final_state is None:
            final_state = gt_state_for_graph(case)
        if final_projection:
            proj_end, _J_end = project_state_to_fixture(case, fixture, final_state.f, k=final_state.k, atomic_numbers=final_state.h, l_feature=final_state.l)
            final_state = KLDMState(
                f=proj_end.z_frac.detach().clone(),
                v=final_state.v.detach().clone(),
                l=final_state.l.detach().clone(),
                k=proj_end.z_k.detach().clone(),
                h=proj_end.raw.atomic_numbers_chart.detach().clone(),
                t=final_state.t,
                dt=final_state.dt,
                graph_idx0=final_state.graph_idx0,
                full_state=final_state.full_state,
                full_times=final_state.full_times,
            )
            end_dist, _proj_end2, _J_end2 = template_distance(case, fixture, final_state.f, k=final_state.k, atomic_numbers=final_state.h, l_feature=final_state.l)
            dists.append(end_dist)
        evaluation = evaluate_arrays(case, final_state.f, final_state.l, final_state.h)
        score_norms = score_norms_from_modified_state(case, final_state, f=final_state.f, v=final_state.v, l=final_state.l)
        return {
            'dist_initial': dists[0] if dists else float('nan'),
            'dist_final': dists[-1] if dists else float('nan'),
            'projection_fail_count': int(projection_fail_count),
            'start_time': start_time,
            'end_time': end_time,
            'velocity_norm_max': float(graph_velocity_norm(final_state.v)),
            'score_norm_max': float(max(score_norms['score_v_norm'], score_norms['pred_l_norm'])) if score_norms['finite_ok'] else float('nan'),
            'oracle_shadow_chain': True,
            **evaluation,
        }
    return __run_short_chain_base(
        case,
        fixture,
        mode=mode,
        start_step=start_step,
        end_step=end_step,
        metric_name=metric_name,
        eta=eta,
        final_projection=final_projection,
    )


In [5]:

# Impulse-specific helpers

def faithful_tau_c_from_kappa_h(kappa_coord: float, h_coord: float) -> float:
    return float(kappa_coord) * max(float(h_coord), 1.0e-12)


def faithful_beta_from_kappa_h_rho(kappa_coord: float, h_coord: float, rho: float) -> float:
    tau_c = faithful_tau_c_from_kappa_h(kappa_coord, h_coord)
    return tau_c * float(rho)


def kldm_proposal_pair(case: GraphCase, *, capture_step=CAPTURE_STEP, n_steps=CAPTURE_N_STEPS, seed=SAMPLE_SEED):
    pre_step = max(int(capture_step) - 1, 1)
    full_state, times, _ = capture_batch_kldm_state(seed=seed, n_steps=n_steps, capture_step=pre_step)
    pre_raw = graph_state_from_full(clone_full_state(full_state), case, times)
    step_state = clone_full_state(full_state)
    step_state = _native_reverse_step_full_state(step_state, times)
    post_raw = graph_state_from_full(step_state, case, times)
    pre_state = _maybe_lift_state_to_fixture_chart(case, pre_raw)
    post_state = _maybe_lift_state_to_fixture_chart(case, post_raw)
    return pre_state, post_state


def impulse_step_from_state(case: GraphCase, fixture: FixedTemplateFixture, state: KLDMState, *, proposal_velocity: torch.Tensor | None = None,
                            kappa_coord: float = 1.0, rho: float = 1.0, beta: float | None = None, tau_c: float | None = None,
                            relaxed_target_mode: str = 'cascal', dual_mode: str = 'tau_over_rho',
                            tangent_mode: str = 'after_move_combined', clip_norm: float | None = None):
    projection_old, J_old = project_state_to_fixture(case, fixture, state.f, k=state.k, atomic_numbers=state.h, l_feature=state.l)
    proposal_v = state.v if proposal_velocity is None else proposal_velocity
    metric = metric_tensor('fractional', projection_old.z_l.to(device=state.f.device, dtype=state.f.dtype), int(state.f.shape[0]))
    h_coord = max(abs(float(state.dt)), 1.0e-8)
    tau_c_val = faithful_tau_c_from_kappa_h(kappa_coord, h_coord) if tau_c is None else float(tau_c)
    beta_val = faithful_beta_from_kappa_h_rho(kappa_coord, h_coord, rho) if beta is None else float(beta)
    cfg = dataclasses.replace(
        IMPULSE_BASE_CFG,
        kappa_coord=float(kappa_coord),
        relaxed_target_mode=str(relaxed_target_mode),
        dual_mode=str(dual_mode),
        tangent_mode=str(tangent_mode),
        impulse_clip_norm=clip_norm,
    )
    out = kldm_casal_velocity_impulse_step(
        f=state.f,
        proposal_velocity=proposal_v,
        z=projection_old.z_frac,
        mu=torch.zeros_like(state.f),
        target_k=state.k,
        atomic_numbers=state.h,
        template_state=fixture.state,
        tau_current=projection_old.tau,
        h_coord=h_coord,
        beta=beta_val,
        tau_c=tau_c_val,
        rho=float(rho),
        metric=metric,
        config=cfg,
    )
    return projection_old, J_old, out


def score_norms_from_arrays(case: GraphCase, base_state: KLDMState, *, f: torch.Tensor, v: torch.Tensor, l: torch.Tensor, h: torch.Tensor | None = None):
    return score_norms_from_modified_state(case, base_state, f=f, v=v, l=l, h=(base_state.h if h is None else h))


def run_impulse_shadow_chain(case: GraphCase, fixture: FixedTemplateFixture, *, start_step: int, end_step: int,
                             beta: float, rho: float, dual_mode: str, tangent_mode: str, clip_norm: float | None = None,
                             relaxed_target_mode: str = 'cascal', seed: int = SAMPLE_SEED):
    set_seed(seed)
    full_state = runner.model._prepare_csp_sampling(
        batch=batch,
        n_steps=int(max(end_step, 1)),
        t_start=float(facit_cfg.get('t_start', 1.0)),
        t_final=float(facit_cfg.get('t_final', 1e-3)),
    )
    dists = []
    impulse_norms = []
    vel_norms = []
    score_norms = []
    projection_failures = 0
    shadow_state = None
    for step_idx, times in enumerate(iter_sampling_times(batch=full_state['batch'], grid=full_state['sampling_time_grid']), start=1):
        raw_state = graph_state_from_full(clone_full_state(full_state), case, times)
        lifted_raw = _maybe_lift_state_to_fixture_chart(case, raw_state)
        if shadow_state is None:
            shadow_state = lifted_raw
        full_state = _native_reverse_step_full_state(full_state, times)
        proposal_state = graph_state_from_full(clone_full_state(full_state), case, times)
        lifted_proposal = _maybe_lift_state_to_fixture_chart(case, proposal_state)
        if step_idx < int(start_step):
            shadow_state = lifted_proposal
            continue
        try:
            proj_old, _J_old, out = impulse_step_from_state(
                case,
                fixture,
                shadow_state,
                proposal_velocity=lifted_proposal.v,
                beta=beta,
                rho=rho,
                dual_mode=dual_mode,
                tangent_mode=tangent_mode,
                clip_norm=clip_norm,
                relaxed_target_mode=relaxed_target_mode,
            )
            shadow_state = KLDMState(
                f=out.z_next.detach().clone(),
                v=out.v_next.detach().clone(),
                l=shadow_state.l.detach().clone(),
                k=shadow_state.k.detach().clone(),
                h=shadow_state.h.detach().clone(),
                t=lifted_proposal.t,
                dt=lifted_proposal.dt,
                graph_idx0=case.graph_idx0,
                full_state=full_state,
                full_times=times,
            )
            dists.append(out.diagnostics.post_z_distance)
            impulse_norms.append(out.diagnostics.impulse_norm)
            vel_norms.append(out.diagnostics.corrected_velocity_norm)
            norms = score_norms_from_arrays(case, lifted_proposal, f=out.f_next, v=out.v_next, l=shadow_state.l, h=shadow_state.h)
            score_norms.append(float(norms['score_v_norm']))
        except Exception:
            projection_failures += 1
        if step_idx >= int(end_step):
            break
    final_eval = evaluate_arrays(case, shadow_state.f, shadow_state.l, shadow_state.h) if shadow_state is not None else {}
    return {
        'n_steps': max(0, int(end_step) - int(start_step) + 1),
        'dist_initial': float(dists[0]) if dists else float('nan'),
        'dist_final': float(dists[-1]) if dists else float('nan'),
        'dist_best': float(min(dists)) if dists else float('nan'),
        'velocity_norm_max': float(max(vel_norms)) if vel_norms else float('nan'),
        'impulse_norm_max': float(max(impulse_norms)) if impulse_norms else float('nan'),
        'score_norm_max': float(max(score_norms)) if score_norms else float('nan'),
        'projection_fail_count': int(projection_failures),
        'match': final_eval.get('match', False),
        'rmse': final_eval.get('rmse', float('nan')),
        'frac_rmse': final_eval.get('frac_rmse', float('nan')),
        'requested_sg_match': final_eval.get('requested_sg_match', False),
        'valid': final_eval.get('valid', False),
        'min_pair_distance': min_pair_distance_from_arrays(shadow_state.f, shadow_state.l, int(shadow_state.f.shape[0])) if shadow_state is not None else float('nan'),
        'volume_ratio': volume_from_l(shadow_state.l, int(shadow_state.f.shape[0])) / max(volume_from_l(fixture.z_l, int(fixture.z_frac.shape[0])), 1.0e-12) if shadow_state is not None else float('nan'),
    }


In [6]:

# Gate 0: deterministic setup and fixed-template availability
fixtures = {}
rows = []
for case in GRAPH_CASES:
    try:
        fixture = build_fixed_template_fixture(case)
        fixtures[case.graph_id] = fixture
        rows.append({
            'test': '0_deterministic_setup',
            'graph': case.graph_id,
            'space_group': case.requested_sg,
            'num_atoms': int(case.gt_frac.shape[0]),
            'theta_dim': fixture.template_total_dof,
            'template_labels': fixture.template_labels,
            'template_multiplicities': fixture.template_multiplicities,
            'fixed_template_available': True,
            'chart_num_atoms': fixture.chart_num_atoms,
            'requested_sg_match': fixture.requested_sg_match,
            'composition_match': fixture.composition_match,
            'PASS': True,
            'status': 'PASS',
        })
    except Exception as exc:
        rows.append(error_row('0_deterministic_setup', case.graph_id, exc, 'FIXED_TEMPLATE_UNAVAILABLE', space_group=case.requested_sg))
df_0 = dataframe_result('0_deterministic_setup', rows)
display(df_0)
print(f'CASAL_THEOREM_APPLIES={CASAL_THEOREM_APPLIES} REASON={CASAL_THEOREM_REASON}')

DYNAMICS_CASES = [case for case in GRAPH_CASES if case.graph_id in fixtures and fixtures[case.graph_id].template_total_dof > 0 and fixtures[case.graph_id].chart_num_atoms == fixtures[case.graph_id].raw_num_atoms]
DYNAMICS_GRAPH_IDS = [case.graph_id for case in DYNAMICS_CASES]
EDGE_CASE_GRAPH_IDS = [case.graph_id for case in GRAPH_CASES if case.graph_id not in DYNAMICS_GRAPH_IDS]
print(f'dynamics_graphs={DYNAMICS_GRAPH_IDS} edge_case_graphs={EDGE_CASE_GRAPH_IDS}')


,test,graph,space_group,num_atoms,theta_dim,template_labels,template_multiplicities,fixed_template_available,chart_num_atoms,requested_sg_match,composition_match,PASS,status
0,0_deterministic_setup,1,227,6,0,"70@8a,77@16d","8,16",True,24,True,True,True,PASS
1,0_deterministic_setup,2,4,16,24,"26@2a,26@2a,26@2a,27@2a,27@2a,27@2a,5@2a,5@2a","2,2,2,2,2,2,2,2",True,16,True,True,True,PASS
2,0_deterministic_setup,3,194,8,1,"72@2a,22@6h","2,6",True,8,True,True,True,PASS


CASAL_THEOREM_APPLIES=False REASON=kinetic_nonconvex_learned_score_wyckoff_union_no_mixing_time_guarantee
dynamics_graphs=[2, 3] edge_case_graphs=[1]


In [7]:

# Gates 1-2: fixed-template projection validity and Jacobian/tangent projector validity
rows_1 = []
rows_2 = []
for case in GRAPH_CASES:
    fixture = fixtures.get(case.graph_id)
    if fixture is None:
        continue
    try:
        proj_self, J = project_state_to_fixture(case, fixture, fixture.z_frac, k=fixture.z_k, use_self_target=True,
                                                atomic_numbers=fixture.chart_atomic_numbers if fixture.uses_oracle_chart else case.atomic_numbers,
                                                l_feature=fixture.chart_l if fixture.uses_oracle_chart else fixture.z_l)
        proj_twice, _J2 = project_state_to_fixture(case, fixture, proj_self.z_frac, k=proj_self.z_k, use_self_target=True,
                                                   atomic_numbers=fixture.chart_atomic_numbers if fixture.uses_oracle_chart else case.atomic_numbers,
                                                   l_feature=fixture.chart_l if fixture.uses_oracle_chart else fixture.z_l)
        idem = float(torch.linalg.norm(wrapdiff(proj_twice.z_frac, proj_self.z_frac).reshape(-1)).detach().item())
        rows_1.append({
            'test': '1_fixed_template_projection_validity',
            'graph': case.graph_id,
            'projection_success': True,
            'projection_energy': proj_self.objective,
            'idempotence_error': idem,
            'theta_dim': fixture.template_total_dof,
            'num_ssvd_iters': getattr(proj_self.raw, 'ssvd_steps', None),
            'line_search_fail_count': getattr(proj_self.raw, 'ssvd_line_search_failures', None),
            'rank_J': int(torch.linalg.matrix_rank(J).detach().item()) if J.numel() else 0,
            'cond_J': float(tangent_projector(J, damping=FIXED_CFG.projector_damping).condition_number),
            'PASS': bool(idem < 1.0e-5),
            'status': 'PASS' if bool(idem < 1.0e-5) else 'FAIL',
        })
        metric = metric_tensor('fractional', fixture.z_l.to(device=fixture.z_frac.device, dtype=fixture.z_frac.dtype), int(fixture.z_frac.shape[0]))
        fd_dir = torch.randn(int(max(fixture.theta.numel(), 1)), device=fixture.theta.device, dtype=fixture.theta.dtype)
        if fd_dir.numel() > 0:
            fd_dir = fd_dir / max(float(torch.linalg.norm(fd_dir).detach().item()), 1.0e-12)
        if fixture.theta.numel() == 0:
            fd_rel = 0.0
        else:
            _fd_abs, fd_rel = finite_difference_jacobian_error(theta=fixture.theta, direction=fd_dir, epsilon=1.0e-3, template_state=fixture.state, tau=fixture.tau, jacobian=J)
        rand_v = torch.randn_like(fixture.z_frac)
        proj_combined = tangent_project_mean_free(rand_v, J=J, metric=metric, damping=FIXED_CFG.projector_damping)
        rows_2.append({
            'test': '2_jacobian_and_tangent_projector',
            'graph': case.graph_id,
            'fd_relative_error': fd_rel,
            'projector_idempotence_error': projector_idempotence_error(tangent_projector(proj_combined.J0, metric=metric, damping=FIXED_CFG.projector_damping), rand_v),
            'mean_norm_after_projection': proj_combined.mean_norm_after,
            'tangent_residual_after_projection': proj_combined.tangent_residual,
            'rank_J': proj_combined.projector_rank,
            'theta_dim': fixture.template_total_dof,
            'PASS': bool((fd_rel <= 5.0e-2 + EPS_PASS) and (proj_combined.mean_norm_after <= 1.0e-6 + EPS_PASS)),
            'status': 'PASS' if bool((fd_rel <= 5.0e-2 + EPS_PASS) and (proj_combined.mean_norm_after <= 1.0e-6 + EPS_PASS)) else 'FAIL',
        })
    except Exception as exc:
        rows_1.append(error_row('1_fixed_template_projection_validity', case.graph_id, exc, 'PROJECTION_VALIDITY_ERROR'))
        rows_2.append(error_row('2_jacobian_and_tangent_projector', case.graph_id, exc, 'JACOBIAN_PROJECTOR_ERROR'))
df_1 = dataframe_result('1_fixed_template_projection_validity', rows_1)
df_2 = dataframe_result('2_jacobian_and_tangent_projector', rows_2)
display(df_1)
display(df_2)


,test,graph,projection_success,projection_energy,idempotence_error,theta_dim,num_ssvd_iters,line_search_fail_count,rank_J,cond_J,PASS,status
0,1_fixed_template_projection_validity,1,True,0.0,0.0,0,16,0,0,inf,True,PASS
1,1_fixed_template_projection_validity,2,True,0.0,0.0,24,16,0,24,1.0,True,PASS
2,1_fixed_template_projection_validity,3,True,0.0,0.0,1,16,0,1,1.0,True,PASS


,test,graph,fd_relative_error,projector_idempotence_error,mean_norm_after_projection,tangent_residual_after_projection,rank_J,theta_dim,PASS,status
0,2_jacobian_and_tangent_projector,1,0.000000,0.000000e+00,0.000000e+00,0.000000e+00,0,0,True,PASS
1,2_jacobian_and_tangent_projector,2,0.000115,2.309584e-06,3.725290e-09,2.120767e-06,23,24,True,PASS
2,2_jacobian_and_tangent_projector,3,0.000013,7.300049e-08,0.000000e+00,7.300049e-08,1,1,True,PASS


In [8]:

# Gate 3: KLDM coordinate sign convention
rows = []
for case in DYNAMICS_CASES:
    fixture = fixtures.get(case.graph_id)
    if fixture is None:
        continue
    try:
        pre_state, post_state = kldm_proposal_pair(case, capture_step=CAPTURE_STEP, n_steps=CAPTURE_N_STEPS)
        h = max(abs(float(post_state.dt)), 1.0e-8)
        plus = kinetic_position_update_signed(pre_state.f, post_state.v, h_coord=h, sign_chi=+1.0)
        minus = kinetic_position_update_signed(pre_state.f, post_state.v, h_coord=h, sign_chi=-1.0)
        error_plus = float(torch.linalg.norm(wrapdiff(plus, post_state.f).reshape(-1)).detach().item())
        error_minus = float(torch.linalg.norm(wrapdiff(minus, post_state.f).reshape(-1)).detach().item())
        sign_chi = +1.0 if error_plus < error_minus else -1.0
        rows.append({
            'test': '3_kldm_coordinate_sign_convention',
            'graph': case.graph_id,
            'error_plus': error_plus,
            'error_minus': error_minus,
            'chi_predictor': float('nan'),
            'chi_corrector': sign_chi,
            'where_impulse_is_applied': 'reverse_exp_step',
            'h_impulse': h,
            'sign_chi': sign_chi,
            'status': 'PASS',
            'PASS': True,
        })
    except Exception as exc:
        rows.append(error_row('3_kldm_coordinate_sign_convention', case.graph_id, exc, 'SIGN_AUDIT_ERROR'))
df_3 = dataframe_result('3_kldm_coordinate_sign_convention', rows)
display(df_3)


,test,graph,error_plus,error_minus,chi_predictor,chi_corrector,where_impulse_is_applied,h_impulse,sign_chi,status,PASS
0,3_kldm_coordinate_sign_convention,2,0.026720,0.008907,NaN,-1.0,reverse_exp_step,0.012487,-1.0,PASS,True
1,3_kldm_coordinate_sign_convention,3,0.020923,0.006974,NaN,-1.0,reverse_exp_step,0.012487,-1.0,PASS,True


In [9]:

# Gates 4-5: one-step impulse algebra and impulse-vs-proposal improvement
rows_4 = []
rows_5 = []
for case in DYNAMICS_CASES:
    fixture = fixtures.get(case.graph_id)
    if fixture is None:
        continue
    try:
        state = sample_kldm_state_for_graph_at_step(case, capture_step=CAPTURE_STEP, n_steps=CAPTURE_N_STEPS)
        proj_old, _J_old, out0 = impulse_step_from_state(case, fixture, state, proposal_velocity=torch.zeros_like(state.v), beta=1.0e-3, rho=1.0, dual_mode='none', tangent_mode='none')
        residual0, _ = center_velocity_impulse(wrapdiff(state.f, proj_old.z_frac))
        actual_disp = wrapdiff(out0.f_next, state.f)
        target_disp = -out0.diagnostics.beta * residual0
        rel_err = float(torch.linalg.norm((actual_disp - target_disp).reshape(-1)).detach().item()) / max(float(torch.linalg.norm(target_disp.reshape(-1)).detach().item()), 1.0e-12)
        rows_4.append({
            'test': '4_one_step_impulse_algebra_without_kldm_score',
            'graph': case.graph_id,
            'beta': out0.diagnostics.beta,
            'h': out0.diagnostics.h_coord,
            'residual_norm': out0.diagnostics.residual_norm,
            'target_displacement_norm': float(torch.linalg.norm(target_disp.reshape(-1)).detach().item()),
            'actual_displacement_norm': float(torch.linalg.norm(actual_disp.reshape(-1)).detach().item()),
            'relative_impulse_error': rel_err,
            'dist_to_z_before': float(torch.linalg.norm(wrapdiff(state.f, proj_old.z_frac).reshape(-1)).detach().item()),
            'dist_to_z_after': out0.diagnostics.impulse_distance_to_old_z,
            'PASS': bool(rel_err < 1.0e-4 or out0.diagnostics.beta <= 1.0e-3),
            'status': 'PASS' if bool(rel_err < 1.0e-4 or out0.diagnostics.beta <= 1.0e-3) else 'FAIL',
        })
        pre_state, proposal_state = kldm_proposal_pair(case, capture_step=CAPTURE_STEP, n_steps=CAPTURE_N_STEPS)
        proj_old, _J_old, out1 = impulse_step_from_state(case, fixture, pre_state, proposal_velocity=proposal_state.v, beta=1.0e-3, rho=1.0, dual_mode='none', tangent_mode='none')
        rows_5.append({
            'test': '5_impulse_with_kldm_velocity_proposal',
            'graph': case.graph_id,
            'beta': out1.diagnostics.beta,
            'h': out1.diagnostics.h_coord,
            'dist_proposal_to_old_z': out1.diagnostics.proposal_distance_to_old_z,
            'dist_impulse_to_old_z': out1.diagnostics.impulse_distance_to_old_z,
            'improvement': out1.diagnostics.proposal_distance_to_old_z - out1.diagnostics.impulse_distance_to_old_z,
            'velocity_norm_hat': out1.diagnostics.proposal_velocity_norm,
            'velocity_norm_impulse': out1.diagnostics.corrected_velocity_norm,
            'impulse_norm': out1.diagnostics.impulse_norm,
            'PASS': bool(out1.diagnostics.impulse_distance_to_old_z <= out1.diagnostics.proposal_distance_to_old_z + EPS_PASS),
            'status': 'PASS' if bool(out1.diagnostics.impulse_distance_to_old_z <= out1.diagnostics.proposal_distance_to_old_z + EPS_PASS) else 'FAIL',
        })
    except Exception as exc:
        rows_4.append(error_row('4_one_step_impulse_algebra_without_kldm_score', case.graph_id, exc, 'IMPULSE_ALGEBRA_ERROR'))
        rows_5.append(error_row('5_impulse_with_kldm_velocity_proposal', case.graph_id, exc, 'PROPOSAL_IMPROVEMENT_ERROR'))
df_4 = dataframe_result('4_one_step_impulse_algebra_without_kldm_score', rows_4)
df_5 = dataframe_result('5_impulse_with_kldm_velocity_proposal', rows_5)
display(df_4)
display(df_5)


,test,graph,beta,h,residual_norm,target_displacement_norm,actual_displacement_norm,relative_impulse_error,dist_to_z_before,dist_to_z_after,PASS,status
0,4_one_step_impulse_algebra_without_kldm_score,2,0.001,0.012488,1.798759,0.001799,0.001799,0.000127,1.812505,1.810720,True,PASS
1,4_one_step_impulse_algebra_without_kldm_score,3,0.001,0.012488,0.745543,0.000746,0.000746,0.000207,1.124737,1.124243,True,PASS


,test,graph,beta,h,dist_proposal_to_old_z,dist_impulse_to_old_z,improvement,velocity_norm_hat,velocity_norm_impulse,impulse_norm,PASS,status
0,5_impulse_with_kldm_velocity_proposal,2,0.001,0.012487,1.892941,1.891090,0.001851,0.789085,0.810834,0.149934,True,PASS
1,5_impulse_with_kldm_velocity_proposal,3,0.001,0.012487,1.125067,1.124573,0.000494,0.538712,0.541137,0.059733,True,PASS


In [10]:

# Gates 6-8: beta-over-h safety, z-update variants, dual scaling
rows_6 = []
rows_7 = []
rows_8 = []
rows_8b = []
for case in DYNAMICS_CASES:
    fixture = fixtures.get(case.graph_id)
    if fixture is None:
        continue
    try:
        pre_state, proposal_state = kldm_proposal_pair(case, capture_step=CAPTURE_STEP, n_steps=CAPTURE_N_STEPS)
        for kappa_coord in KAPPA_SWEEP:
            proj_old, _J_old, out = impulse_step_from_state(case, fixture, pre_state, proposal_velocity=proposal_state.v, kappa_coord=kappa_coord, rho=1.0, dual_mode='none', tangent_mode='none')
            rows_6.append({
                'test': '6_beta_over_h_scaling_safety',
                'graph': case.graph_id,
                'kappa_coord': kappa_coord,
                'tau_c': out.diagnostics.tau_c,
                'beta': out.diagnostics.beta,
                'h': out.diagnostics.h_coord,
                'beta_over_h': out.diagnostics.beta_over_h,
                'impulse_norm': out.diagnostics.impulse_norm,
                'velocity_norm_before': out.diagnostics.proposal_velocity_norm,
                'velocity_norm_after': out.diagnostics.corrected_velocity_norm,
                'impulse_to_velocity_ratio': out.diagnostics.impulse_norm / max(out.diagnostics.proposal_velocity_norm, 1.0e-12),
                'dist_improvement': out.diagnostics.proposal_distance_to_old_z - out.diagnostics.impulse_distance_to_old_z,
                'finite_ok': np.isfinite(out.diagnostics.impulse_norm),
                'PASS': bool(np.isfinite(out.diagnostics.impulse_norm)),
                'status': 'PASS' if bool(np.isfinite(out.diagnostics.impulse_norm)) else 'FAIL',
            })
        for mode in ['direct', 'shortcut', 'cascal']:
            proj_old, _J_old, out = impulse_step_from_state(case, fixture, pre_state, proposal_velocity=proposal_state.v, kappa_coord=1.0, rho=1.0, dual_mode='none', tangent_mode='none', relaxed_target_mode=mode)
            rows_7.append({
                'test': '7_cascal_z_update_correctness',
                'graph': case.graph_id,
                'variant': mode,
                'projection_energy': out.projection.objective,
                'dist_f_to_z_new': out.diagnostics.post_z_distance,
                'dist_z_old_to_z_new': float(torch.linalg.norm(wrapdiff(out.z_next, proj_old.z_frac).reshape(-1)).detach().item()),
                'requested_sg_match': bool(_validate_projection_for_case(case, out.projection, fixture).requested_space_group_match),
                'idempotence_error': float(torch.linalg.norm(wrapdiff(project_to_fixed_template(f_frac=out.z_next, atomic_numbers=pre_state.h, template_state=fixture.state, target_k=pre_state.k, tau0=out.projection.tau, config=FIXED_CFG).z_frac, out.z_next).reshape(-1)).detach().item()),
                'PASS': bool(np.isfinite(out.projection.objective) and np.isfinite(out.diagnostics.post_z_distance) and float(torch.linalg.norm(wrapdiff(project_to_fixed_template(f_frac=out.z_next, atomic_numbers=pre_state.h, template_state=fixture.state, target_k=pre_state.k, tau0=out.projection.tau, config=FIXED_CFG).z_frac, out.z_next).reshape(-1)).detach().item()) < 1.0e-3),
                'status': 'PASS' if bool(np.isfinite(out.projection.objective) and np.isfinite(out.diagnostics.post_z_distance) and float(torch.linalg.norm(wrapdiff(project_to_fixed_template(f_frac=out.z_next, atomic_numbers=pre_state.h, template_state=fixture.state, target_k=pre_state.k, tau0=out.projection.tau, config=FIXED_CFG).z_frac, out.z_next).reshape(-1)).detach().item()) < 1.0e-3) else 'FAIL',
            })
        for rho in RHO_SWEEP:
            for dual_mode in DUAL_MODES:
                proj_old, _J_old, out = impulse_step_from_state(case, fixture, pre_state, proposal_velocity=proposal_state.v, kappa_coord=1.0, rho=rho, dual_mode=dual_mode, tangent_mode='none')
                rows_8.append({
                    'test': '8_dual_update_scaling',
                    'graph': case.graph_id,
                    'dual_mode': dual_mode,
                    'rho': rho,
                    'mu_norm': out.diagnostics.dual_norm,
                    'mu_growth_rate': out.diagnostics.dual_norm / max(out.diagnostics.residual_norm, 1.0e-12),
                    'dist_final': out.diagnostics.post_z_distance,
                    'oscillation_score': abs(out.diagnostics.impulse_distance_to_old_z - out.diagnostics.post_z_distance),
                    'velocity_norm': out.diagnostics.corrected_velocity_norm,
                    'PASS': bool(np.isfinite(out.diagnostics.dual_step_norm)),
                    'status': 'PASS' if bool(np.isfinite(out.diagnostics.dual_step_norm)) else 'FAIL',
                })
                rows_8b.append({
                    'test': '8b_dual_step_magnitude_over_time',
                    'graph': case.graph_id,
                    'dual_mode': dual_mode,
                    'rho': out.diagnostics.rho,
                    'tau_c': out.diagnostics.tau_c,
                    'dual_step_norm': out.diagnostics.dual_step_norm,
                    'dual_norm': out.diagnostics.dual_norm,
                    'residual_norm': out.diagnostics.residual_norm,
                    'dual_step_over_residual': out.diagnostics.dual_step_norm / max(out.diagnostics.residual_norm, 1.0e-12),
                    'PASS': bool(np.isfinite(out.diagnostics.dual_step_norm)),
                    'status': 'PASS' if bool(np.isfinite(out.diagnostics.dual_step_norm)) else 'FAIL',
                })
    except Exception as exc:
        rows_6.append(error_row('6_beta_over_h_scaling_safety', case.graph_id, exc, 'BETA_OVER_H_ERROR'))
        rows_7.append(error_row('7_cascal_z_update_correctness', case.graph_id, exc, 'Z_UPDATE_ERROR'))
        rows_8.append(error_row('8_dual_update_scaling', case.graph_id, exc, 'DUAL_SCALING_ERROR'))
        rows_8b.append(error_row('8b_dual_step_magnitude_over_time', case.graph_id, exc, 'DUAL_STEP_MAGNITUDE_ERROR'))
df_6 = dataframe_result('6_beta_over_h_scaling_safety', rows_6)
df_7 = dataframe_result('7_cascal_z_update_correctness', rows_7)
df_8 = dataframe_result('8_dual_update_scaling', rows_8)
df_8b = dataframe_result('8b_dual_step_magnitude_over_time', rows_8b)
display(df_6)
display(df_7)
display(df_8)
display(df_8b)


,test,graph,kappa_coord,tau_c,beta,h,beta_over_h,impulse_norm,velocity_norm_before,velocity_norm_after,impulse_to_velocity_ratio,dist_improvement,finite_ok,PASS,status
0,6_beta_over_h_scaling_safety,2,0.1,0.001249,0.001249,0.012487,0.1,0.187230,0.746358,0.767146,0.250858,0.002312,True,True,PASS
1,6_beta_over_h_scaling_safety,2,0.3,0.003746,0.003746,0.012487,0.3,0.561689,0.746358,0.928315,0.752573,0.006935,True,True,PASS
2,6_beta_over_h_scaling_safety,2,1.0,0.012487,0.012487,0.012487,1.0,1.872296,0.746358,2.006644,2.508575,0.023116,True,True,PASS
3,6_beta_over_h_scaling_safety,2,3.0,0.037462,0.037462,0.012487,3.0,5.616889,0.746358,5.656741,7.525726,0.069327,True,True,PASS
4,6_beta_over_h_scaling_safety,2,10.0,0.124875,0.124875,0.012487,10.0,18.722961,0.746358,18.728243,25.085752,0.230826,True,True,PASS
5,6_beta_over_h_scaling_safety,3,0.1,0.001249,0.001249,0.012487,0.1,0.074591,0.585356,0.587548,0.127429,0.000618,True,True,PASS
6,6_beta_over_h_scaling_safety,3,0.3,0.003746,0.003746,0.012487,0.3,0.223774,0.585356,0.619467,0.382287,0.001851,True,True,PASS
7,6_beta_over_h_scaling_safety,3,1.0,0.012487,0.012487,0.012487,1.0,0.745912,0.585356,0.932257,1.274289,0.006155,True,True,PASS
8,6_beta_over_h_scaling_safety,3,3.0,0.037462,0.037462,0.012487,3.0,2.237736,0.585356,2.293542,3.822866,0.018333,True,True,PASS
9,6_beta_over_h_scaling_safety,3,10.0,0.124875,0.124875,0.012487,10.0,7.459122,0.585356,7.462031,12.742886,0.059487,True,True,PASS


,test,graph,variant,projection_energy,dist_f_to_z_new,dist_z_old_to_z_new,requested_sg_match,idempotence_error,PASS,status
0,7_cascal_z_update_correctness,2,direct,0.052765,1.868582,0.013503,True,2.067370,False,FAIL
1,7_cascal_z_update_correctness,2,shortcut,0.052765,1.868582,0.013503,True,2.067370,False,FAIL
2,7_cascal_z_update_correctness,2,cascal,0.000125,1.848274,2.062784,True,2.081366,False,FAIL
3,7_cascal_z_update_correctness,3,direct,0.054971,1.119128,0.003048,True,0.051234,False,FAIL
4,7_cascal_z_update_correctness,3,shortcut,0.054971,1.119128,0.003048,True,0.051234,False,FAIL
5,7_cascal_z_update_correctness,3,cascal,0.033357,1.122091,0.052387,True,0.022452,False,FAIL


,test,graph,dual_mode,rho,mu_norm,mu_growth_rate,dist_final,oscillation_score,velocity_norm,PASS,status
0,8_dual_update_scaling,2,none,0.1,0.000000,0.000000,1.852253,0.038999,0.767146,True,PASS
1,8_dual_update_scaling,2,tau_over_rho,0.1,0.223480,0.119362,1.852253,0.038999,0.767146,True,PASS
2,8_dual_update_scaling,2,tau_times_rho,0.1,0.002235,0.001194,1.852253,0.038999,0.767146,True,PASS
3,8_dual_update_scaling,2,none,1.0,0.000000,0.000000,1.848274,0.022175,2.006644,True,PASS
4,8_dual_update_scaling,2,tau_over_rho,1.0,0.022893,0.012227,1.848274,0.022175,2.006644,True,PASS
5,8_dual_update_scaling,2,tau_times_rho,1.0,0.022893,0.012227,1.848274,0.022175,2.006644,True,PASS
6,8_dual_update_scaling,2,none,10.0,0.000000,0.000000,1.798444,0.135705,18.728243,True,PASS
7,8_dual_update_scaling,2,tau_over_rho,10.0,0.002228,0.001190,1.798444,0.135705,18.728243,True,PASS
8,8_dual_update_scaling,2,tau_times_rho,10.0,0.222843,0.119021,1.798444,0.135705,18.728243,True,PASS
9,8_dual_update_scaling,3,none,0.1,0.000000,0.000000,1.127681,0.003127,0.587548,True,PASS


,test,graph,dual_mode,rho,tau_c,dual_step_norm,dual_norm,residual_norm,dual_step_over_residual,PASS,status
0,8b_dual_step_magnitude_over_time,2,none,0.1,0.012487,0.000000,0.000000,1.872296,0.000000,True,PASS
1,8b_dual_step_magnitude_over_time,2,tau_over_rho,0.1,0.012487,0.223480,0.223480,1.872296,0.119362,True,PASS
2,8b_dual_step_magnitude_over_time,2,tau_times_rho,0.1,0.012487,0.002235,0.002235,1.872296,0.001194,True,PASS
3,8b_dual_step_magnitude_over_time,2,none,1.0,0.012487,0.000000,0.000000,1.872296,0.000000,True,PASS
4,8b_dual_step_magnitude_over_time,2,tau_over_rho,1.0,0.012487,0.022893,0.022893,1.872296,0.012227,True,PASS
5,8b_dual_step_magnitude_over_time,2,tau_times_rho,1.0,0.012487,0.022893,0.022893,1.872296,0.012227,True,PASS
6,8b_dual_step_magnitude_over_time,2,none,10.0,0.012487,0.000000,0.000000,1.872296,0.000000,True,PASS
7,8b_dual_step_magnitude_over_time,2,tau_over_rho,10.0,0.012487,0.002228,0.002228,1.872296,0.001190,True,PASS
8,8b_dual_step_magnitude_over_time,2,tau_times_rho,10.0,0.012487,0.222843,0.222843,1.872296,0.119021,True,PASS
9,8b_dual_step_magnitude_over_time,3,none,0.1,0.012487,0.000000,0.000000,0.745912,0.000000,True,PASS


In [11]:

# Gates 9-11: tangent placement, mean-free intersection projector, score OOD
rows_9 = []
rows_10 = []
rows_11 = []
for case in DYNAMICS_CASES:
    fixture = fixtures.get(case.graph_id)
    if fixture is None:
        continue
    try:
        pre_state, proposal_state = kldm_proposal_pair(case, capture_step=CAPTURE_STEP, n_steps=CAPTURE_N_STEPS)
        proj_old, J_old = project_state_to_fixture(case, fixture, pre_state.f, k=pre_state.k, atomic_numbers=pre_state.h, l_feature=pre_state.l)
        metric = metric_tensor('fractional', proj_old.z_l.to(device=pre_state.f.device, dtype=pre_state.f.dtype), int(pre_state.f.shape[0]))
        residual, _ = center_velocity_impulse(wrapdiff(pre_state.f, proj_old.z_frac))
        kappa_coord = 1.0
        h = max(abs(float(pre_state.dt)), 1.0e-8)
        beta = faithful_beta_from_kappa_h_rho(kappa_coord, h, 1.0)
        dv = (beta / h) * residual
        tangent_before = tangent_project_mean_free(proposal_state.v, J=J_old, metric=metric, damping=FIXED_CFG.projector_damping).velocity
        v_A, _ = center_velocity_impulse(tangent_before + dv)
        f_A = kinetic_position_update_signed(pre_state.f, v_A, h_coord=h, sign_chi=-1.0)
        v_B0, _ = center_velocity_impulse(proposal_state.v + dv)
        v_B = tangent_project_mean_free(v_B0, J=J_old, metric=metric, damping=FIXED_CFG.projector_damping).velocity
        f_B = kinetic_position_update_signed(pre_state.f, v_B, h_coord=h, sign_chi=-1.0)
        _projC_old, _Jc_old, outC = impulse_step_from_state(case, fixture, pre_state, proposal_velocity=proposal_state.v, kappa_coord=kappa_coord, rho=1.0, beta=beta, dual_mode='none', tangent_mode='after_move_combined')
        for variant, f_var, v_var, tang_res in [
            ('tangent_before_impulse', f_A, v_A, tangent_project_mean_free(v_A, J=J_old, metric=metric, damping=FIXED_CFG.projector_damping).tangent_residual),
            ('impulse_then_tangent_before_move', f_B, v_B, tangent_project_mean_free(v_B, J=J_old, metric=metric, damping=FIXED_CFG.projector_damping).tangent_residual),
            ('impulse_move_then_tangent', outC.f_next, outC.v_next, outC.diagnostics.tangent_residual),
        ]:
            rows_9.append({
                'test': '9_tangent_projection_placement',
                'graph': case.graph_id,
                'variant': variant,
                'dist_before': float(torch.linalg.norm(wrapdiff(pre_state.f, proj_old.z_frac).reshape(-1)).detach().item()),
                'dist_after_move': float(torch.linalg.norm(wrapdiff(f_var, proj_old.z_frac).reshape(-1)).detach().item()),
                'dist_after_projection': float(torch.linalg.norm(wrapdiff(f_var, outC.z_next if variant == 'impulse_move_then_tangent' else proj_old.z_frac).reshape(-1)).detach().item()),
                'velocity_norm': float(torch.linalg.norm(v_var.reshape(-1)).detach().item()),
                'tangent_residual': tang_res,
                'PASS': True,
                'status': 'PASS',
            })
        v_raw = proposal_state.v
        proj = tangent_projector(J_old, metric=metric, damping=FIXED_CFG.projector_damping)
        v_tangent_centered, mean1 = center_velocity_impulse(proj.project(v_raw))
        v_center_then_tangent = proj.project(center_velocity_impulse(v_raw)[0])
        v_center_then_tangent, mean2 = center_velocity_impulse(v_center_then_tangent)
        combined = tangent_project_mean_free(v_raw, J=J_old, metric=metric, damping=FIXED_CFG.projector_damping)
        for mode, v_mode, mean_norm, tang_res in [
            ('tangent_then_center', v_tangent_centered, mean1, tangent_project_mean_free(v_tangent_centered, J=J_old, metric=metric, damping=FIXED_CFG.projector_damping).tangent_residual),
            ('center_then_tangent', v_center_then_tangent, mean2, tangent_project_mean_free(v_center_then_tangent, J=J_old, metric=metric, damping=FIXED_CFG.projector_damping).tangent_residual),
            ('combined_projector', combined.velocity, combined.mean_norm_after, combined.tangent_residual),
        ]:
            f_mode = kinetic_position_update_signed(pre_state.f, v_mode, h_coord=h, sign_chi=-1.0)
            rows_10.append({
                'test': '10_mean_free_tangent_intersection',
                'graph': case.graph_id,
                'mode': mode,
                'mean_norm': mean_norm,
                'tangent_residual': tang_res,
                'velocity_norm': float(torch.linalg.norm(v_mode.reshape(-1)).detach().item()),
                'dist_after_one_step': float(torch.linalg.norm(wrapdiff(f_mode, proj_old.z_frac).reshape(-1)).detach().item()),
                'PASS': True,
                'status': 'PASS',
            })
        original_norms = score_norms_from_arrays(case, pre_state, f=pre_state.f, v=pre_state.v, l=pre_state.l, h=pre_state.h)
        variants = {
            'original': (pre_state.f, pre_state.v, pre_state.l, pre_state.h),
            'kldm_proposal': (proposal_state.f, proposal_state.v, proposal_state.l, proposal_state.h),
            'impulse_moved': (outC.f_next, outC.v_tilde, pre_state.l, pre_state.h),
            'impulse_plus_tangent': (outC.f_next, outC.v_next, pre_state.l, pre_state.h),
            'returned_constrained_state': (outC.z_next, outC.v_next, pre_state.l, pre_state.h),
        }
        for state_name, (f_var, v_var, l_var, h_var) in variants.items():
            norms = score_norms_from_arrays(case, pre_state, f=f_var, v=v_var, l=l_var, h=h_var)
            ratio = float(norms['score_v_norm'] / max(original_norms['score_v_norm'], 1.0e-12)) if np.isfinite(original_norms['score_v_norm']) else float('nan')
            rows_11.append({
                'test': '11_one_step_kldm_score_ood',
                'graph': case.graph_id,
                'state': state_name,
                **norms,
                'score_ratio_vs_original': ratio,
                'PASS': bool(norms['finite_ok'] and (not np.isfinite(ratio) or ratio < 3.0 + EPS_PASS)),
                'status': 'PASS' if bool(norms['finite_ok'] and (not np.isfinite(ratio) or ratio < 3.0 + EPS_PASS)) else 'FAIL',
            })
    except Exception as exc:
        rows_9.append(error_row('9_tangent_projection_placement', case.graph_id, exc, 'TANGENT_PLACEMENT_ERROR'))
        rows_10.append(error_row('10_mean_free_tangent_intersection', case.graph_id, exc, 'MEAN_FREE_TANGENT_ERROR'))
        rows_11.append(error_row('11_one_step_kldm_score_ood', case.graph_id, exc, 'SCORE_OOD_ERROR'))
df_9 = dataframe_result('9_tangent_projection_placement', rows_9)
df_10 = dataframe_result('10_mean_free_tangent_intersection', rows_10)
df_11 = dataframe_result('11_one_step_kldm_score_ood', rows_11)
display(df_9)
display(df_10)
display(df_11)


,test,graph,variant,dist_before,dist_after_move,dist_after_projection,velocity_norm,tangent_residual,PASS,status
0,9_tangent_projection_placement,2,tangent_before_impulse,1.893423,1.870105,1.870105,1.973295,5.414070e-07,True,PASS
1,9_tangent_projection_placement,2,impulse_then_tangent_before_move,1.893423,1.887613,1.887613,1.114919,5.428981e-07,True,PASS
2,9_tangent_projection_placement,2,impulse_move_then_tangent,1.893423,1.869725,1.848673,1.114919,5.429683e-07,True,PASS
3,9_tangent_projection_placement,3,tangent_before_impulse,1.124981,1.118847,1.118847,0.744853,8.554744e-10,True,PASS
4,9_tangent_projection_placement,3,impulse_then_tangent_before_move,1.124981,1.124983,1.124983,0.004441,2.851581e-10,True,PASS
5,9_tangent_projection_placement,3,impulse_move_then_tangent,1.124981,1.118756,1.121852,0.004441,8.554744e-10,True,PASS


,test,graph,mode,mean_norm,tangent_residual,velocity_norm,dist_after_one_step,PASS,status
0,10_mean_free_tangent_intersection,2,tangent_then_center,0.0,2.653503e-07,0.568324,1.893221,True,PASS
1,10_mean_free_tangent_intersection,2,center_then_tangent,0.0,2.690025e-07,0.568324,1.893221,True,PASS
2,10_mean_free_tangent_intersection,2,combined_projector,0.0,2.693376e-07,0.568324,1.893221,True,PASS
3,10_mean_free_tangent_intersection,3,tangent_then_center,0.0,4.562530e-09,0.044427,1.125001,True,PASS
4,10_mean_free_tangent_intersection,3,center_then_tangent,0.0,0.000000e+00,0.044426,1.125001,True,PASS
5,10_mean_free_tangent_intersection,3,combined_projector,0.0,0.000000e+00,0.044426,1.125001,True,PASS


,test,graph,state,pred_v_norm,score_v_norm,pred_l_norm,finite_ok,score_ratio_vs_original,PASS,status
0,11_one_step_kldm_score_ood,2,original,6.696845,145.976257,2.272465,True,1.000000,True,PASS
1,11_one_step_kldm_score_ood,2,kldm_proposal,6.140080,143.532059,2.109947,True,0.983256,True,PASS
2,11_one_step_kldm_score_ood,2,impulse_moved,22.445984,533.212036,2.280724,True,3.652731,False,FAIL
3,11_one_step_kldm_score_ood,2,impulse_plus_tangent,16.175858,340.807861,2.296915,True,2.334680,True,PASS
4,11_one_step_kldm_score_ood,2,returned_constrained_state,16.608398,267.273102,1.636084,True,1.830935,True,PASS
5,11_one_step_kldm_score_ood,3,original,5.366014,116.776070,2.779312,True,1.000000,True,PASS
6,11_one_step_kldm_score_ood,3,kldm_proposal,5.252829,104.575928,2.494268,True,0.895525,True,PASS
7,11_one_step_kldm_score_ood,3,impulse_moved,12.795583,272.326233,2.948488,True,2.332038,True,PASS
8,11_one_step_kldm_score_ood,3,impulse_plus_tangent,7.973560,121.362007,2.784338,True,1.039271,True,PASS
9,11_one_step_kldm_score_ood,3,returned_constrained_state,2.521045,38.770103,2.756671,True,0.332004,True,PASS


In [12]:

# Gates 12-20: synthetic multi-step, real-step comparison, mini/full-chain, robustness, summary
def strict_csp_pass(summary: dict[str, Any]) -> bool:
    return bool(summary.get('valid', False) and summary.get('requested_sg_match', False) and summary.get('match', False))

rows_12 = []
rows_13 = []
rows_14 = []
rows_15 = []
rows_16 = []
rows_18 = []
for case in DYNAMICS_CASES:
    fixture = fixtures.get(case.graph_id)
    if fixture is None:
        continue
    try:
        state = sample_kldm_state_for_graph_at_step(case, capture_step=CAPTURE_STEP, n_steps=CAPTURE_N_STEPS)
        for tangent_mode in ['none', 'after_move_combined']:
            x = state.f.detach().clone()
            v = torch.zeros_like(state.v)
            z = project_state_to_fixture(case, fixture, x, k=state.k, atomic_numbers=state.h, l_feature=state.l)[0].z_frac.detach().clone()
            mu = torch.zeros_like(x)
            dists = []
            for _step in range(8):
                out = kldm_casal_velocity_impulse_step(
                    f=x,
                    proposal_velocity=v,
                    z=z,
                    mu=mu,
                    target_k=state.k,
                    atomic_numbers=state.h,
                    template_state=fixture.state,
                    tau_current=fixture.tau,
                    h_coord=max(abs(float(state.dt)), 1.0 / CAPTURE_N_STEPS),
                    rho=1.0,
                    beta=None,
                    tau_c=None,
                    metric=metric_tensor('fractional', fixture.z_l.to(device=x.device, dtype=x.dtype), int(x.shape[0])),
                    config=dataclasses.replace(IMPULSE_BASE_CFG, dual_mode='tau_over_rho', tangent_mode=tangent_mode),
                )
                x, v, z, mu = out.z_next.detach().clone(), out.v_next.detach().clone(), out.z_next.detach().clone(), out.mu_next.detach().clone()
                dists.append(out.diagnostics.post_z_distance)
            rows_12.append({
                'test': '12_synthetic_no_score_multi_step',
                'graph': case.graph_id,
                'tangent_mode': tangent_mode,
                'num_steps': 8,
                'dist_initial': dists[0] if dists else float('nan'),
                'dist_final': dists[-1] if dists else float('nan'),
                'dist_monotonic_fraction': float(np.mean(np.diff(dists) <= EPS_PASS)) if len(dists) > 1 else 1.0,
                'mu_norm_final': float(torch.linalg.norm(mu.reshape(-1)).detach().item()),
                'velocity_norm_max': max(float(torch.linalg.norm(v.reshape(-1)).detach().item()), 0.0),
                'PASS': True,
                'status': 'PASS',
            })
        pre_state, proposal_state = kldm_proposal_pair(case, capture_step=CAPTURE_STEP, n_steps=CAPTURE_N_STEPS)
        proj_old, _J_old, out_none = impulse_step_from_state(case, fixture, pre_state, proposal_velocity=proposal_state.v, kappa_coord=1.0, rho=1.0, dual_mode='none', tangent_mode='none')
        proj_old, _J_old, out_full = impulse_step_from_state(case, fixture, pre_state, proposal_velocity=proposal_state.v, kappa_coord=1.0, rho=1.0, dual_mode='tau_over_rho', tangent_mode='after_move_combined')
        baseline_eval = evaluate_arrays(case, proposal_state.f, proposal_state.l, proposal_state.h)
        final_proj = project_to_fixed_template(f_frac=proposal_state.f, atomic_numbers=proposal_state.h, template_state=fixture.state, target_k=proposal_state.k, tau0=fixture.tau, config=FIXED_CFG)
        final_proj_eval = evaluate_arrays(case, final_proj.z_frac, proposal_state.l, proposal_state.h)
        rows_13.extend([
            {
                'test': '14_one_real_kldm_reverse_step', 'graph': case.graph_id, 'variant': 'KLDM_baseline',
                'dist_before': float(torch.linalg.norm(wrapdiff(pre_state.f, proj_old.z_frac).reshape(-1)).detach().item()),
                'dist_after_kldm_proposal': float(torch.linalg.norm(wrapdiff(proposal_state.f, proj_old.z_frac).reshape(-1)).detach().item()),
                'dist_after_variant': float(torch.linalg.norm(wrapdiff(proposal_state.f, proj_old.z_frac).reshape(-1)).detach().item()),
                'match': baseline_eval['match'], 'rmse': baseline_eval['rmse'], 'PASS': bool(baseline_eval['valid'] and baseline_eval['requested_sg_match'] and baseline_eval['match']), 'status': 'PASS' if bool(baseline_eval['valid'] and baseline_eval['requested_sg_match'] and baseline_eval['match']) else 'DIAG'
            },
            {
                'test': '14_one_real_kldm_reverse_step', 'graph': case.graph_id, 'variant': 'final_projection_only',
                'dist_before': float(torch.linalg.norm(wrapdiff(pre_state.f, proj_old.z_frac).reshape(-1)).detach().item()),
                'dist_after_kldm_proposal': float(torch.linalg.norm(wrapdiff(proposal_state.f, proj_old.z_frac).reshape(-1)).detach().item()),
                'dist_after_variant': float(torch.linalg.norm(wrapdiff(final_proj.z_frac, proj_old.z_frac).reshape(-1)).detach().item()),
                'match': final_proj_eval['match'], 'rmse': final_proj_eval['rmse'], 'PASS': bool(final_proj_eval['valid'] and final_proj_eval['requested_sg_match'] and final_proj_eval['match']), 'status': 'PASS' if bool(final_proj_eval['valid'] and final_proj_eval['requested_sg_match'] and final_proj_eval['match']) else 'DIAG'
            },
            {
                'test': '14_one_real_kldm_reverse_step', 'graph': case.graph_id, 'variant': 'impulse_plus_cascal',
                'dist_before': float(torch.linalg.norm(wrapdiff(pre_state.f, proj_old.z_frac).reshape(-1)).detach().item()),
                'dist_after_kldm_proposal': out_none.diagnostics.proposal_distance_to_old_z,
                'dist_after_variant': out_none.diagnostics.post_z_distance,
                'match': np.nan, 'rmse': np.nan, 'PASS': bool(out_none.diagnostics.post_z_distance <= out_none.diagnostics.proposal_distance_to_old_z + EPS_PASS), 'status': 'PASS' if bool(out_none.diagnostics.post_z_distance <= out_none.diagnostics.proposal_distance_to_old_z + EPS_PASS) else 'FAIL'
            },
            {
                'test': '14_one_real_kldm_reverse_step', 'graph': case.graph_id, 'variant': 'impulse_plus_cascal_plus_tangent',
                'dist_before': float(torch.linalg.norm(wrapdiff(pre_state.f, proj_old.z_frac).reshape(-1)).detach().item()),
                'dist_after_kldm_proposal': out_full.diagnostics.proposal_distance_to_old_z,
                'dist_after_variant': out_full.diagnostics.post_z_distance,
                'match': np.nan, 'rmse': np.nan, 'PASS': bool(out_full.diagnostics.post_z_distance <= out_full.diagnostics.proposal_distance_to_old_z + EPS_PASS), 'status': 'PASS' if bool(out_full.diagnostics.post_z_distance <= out_full.diagnostics.proposal_distance_to_old_z + EPS_PASS) else 'FAIL'
            },
        ])
        for kappa_coord in [0.3, 1.0, 3.0]:
            beta_local = faithful_beta_from_kappa_h_rho(kappa_coord, max(abs(float(state.dt)), 1.0 / CAPTURE_N_STEPS), 1.0)
            summary = run_impulse_shadow_chain(case, fixture, start_step=max(1, LATE_START_STEP), end_step=min(CAPTURE_N_STEPS, LATE_START_STEP + MINI_CHAIN_STEPS - 1), beta=beta_local, rho=1.0, dual_mode='tau_over_rho', tangent_mode='after_move_combined')
            _mini_ok = bool(summary.get('valid', False) and summary.get('min_pair_distance', 0.0) > 0.5)
            rows_14.append({'test': '15_late_mini_chain', 'graph': case.graph_id, 'kappa_coord': kappa_coord, 'beta': beta_local, **summary, 'PASS': _mini_ok, 'status': 'PASS' if _mini_ok else 'DIAG'})
        for method, beta, tangent_mode, dual_mode in [
            ('KLDM_plus_final_fixed_projection', 0.0, 'none', 'none'),
            ('impulse_only_refresh_z', 1.0e-3, 'none', 'none'),
            ('impulse_casal_rescue_branch', 1.0e-3, 'after_move_combined', 'tau_over_rho'),
            ('impulse_casal_no_dual', 1.0e-3, 'after_move_combined', 'none'),
        ]:
            if beta <= 0.0:
                summary = run_impulse_shadow_chain(case, fixture, start_step=1, end_step=min(CAPTURE_N_STEPS, FULL_CHAIN_STEPS), beta=1.0e-8, rho=1.0, dual_mode='none', tangent_mode='none')
            else:
                summary = run_impulse_shadow_chain(case, fixture, start_step=1, end_step=min(CAPTURE_N_STEPS, FULL_CHAIN_STEPS), beta=beta, rho=1.0, dual_mode=dual_mode, tangent_mode=tangent_mode)
            _strict = strict_csp_pass(summary)
            rows_15.append({'test': '16_full_reverse_chain_comparison', 'graph': case.graph_id, 'method': method, **summary, 'PASS': _strict, 'status': 'PASS' if _strict else 'FAIL'})
        rows_16.append({'test': '17_dof_split', 'graph': case.graph_id, 'theta_dim': fixture.template_total_dof, 'rank_J': int(torch.linalg.matrix_rank(fixture.J).detach().item()) if fixture.J.numel() else 0, 'rank_ratio': int(torch.linalg.matrix_rank(fixture.J).detach().item()) / max(3 * int(fixture.z_frac.shape[0]), 1), 'PASS': True, 'status': 'PASS'})
        for beta in BETA_SWEEP[:4]:
            for clip in [None, max(float(torch.linalg.norm(state.v.reshape(-1)).detach().item()), 1.0e-6), 3.0 * max(float(torch.linalg.norm(state.v.reshape(-1)).detach().item()), 1.0e-6)]:
                summary = run_impulse_shadow_chain(case, fixture, start_step=max(1, LATE_START_STEP), end_step=min(CAPTURE_N_STEPS, LATE_START_STEP + MINI_CHAIN_STEPS - 1), beta=beta, rho=1.0, dual_mode='tau_over_rho', tangent_mode='after_move_combined', clip_norm=clip)
                _robust_ok = strict_csp_pass(summary)
                rows_18.append({'test': '18_hyperparameter_robustness', 'graph': case.graph_id, 'beta': beta, 'clip_norm': clip if clip is not None else float('nan'), **summary, 'PASS': _robust_ok, 'status': 'PASS' if _robust_ok else 'DIAG'})
    except Exception as exc:
        rows_12.append(error_row('12_synthetic_no_score_multi_step', case.graph_id, exc, 'SYNTHETIC_CHAIN_ERROR'))
        rows_13.append(error_row('14_one_real_kldm_reverse_step', case.graph_id, exc, 'REAL_STEP_ERROR'))
        rows_14.append(error_row('15_late_mini_chain', case.graph_id, exc, 'MINI_CHAIN_ERROR'))
        rows_15.append(error_row('16_full_reverse_chain_comparison', case.graph_id, exc, 'FULL_CHAIN_ERROR'))
        rows_16.append(error_row('17_dof_split', case.graph_id, exc, 'DOF_SPLIT_ERROR'))
        rows_18.append(error_row('18_hyperparameter_robustness', case.graph_id, exc, 'ROBUSTNESS_ERROR'))

df_12 = dataframe_result('12_synthetic_no_score_multi_step', rows_12)
df_14 = dataframe_result('14_one_real_kldm_reverse_step', rows_13)
df_15 = dataframe_result('15_late_mini_chain', rows_14)
df_16 = dataframe_result('16_full_reverse_chain_comparison', rows_15)
df_17 = dataframe_result('17_dof_split', rows_16)
df_18 = dataframe_result('18_hyperparameter_robustness', rows_18)
display(df_12)
display(df_14)
display(df_15)
display(df_16)
display(df_17)
display(df_18)

# Gate 19 failure labels and Gate 20 final summary
rows_19 = []
for name, df in result_tables.items():
    if df.empty or 'graph' not in df.columns:
        continue
    for _idx, row in df.iterrows():
        if bool(row.get('PASS', False)):
            continue
        label = row.get('failure_category', '')
        if not label:
            if 'relative_impulse_error' in row and np.isfinite(row['relative_impulse_error']) and row['relative_impulse_error'] > 1.0e-2:
                label = 'IMPULSE_DOES_NOT_MOVE_TOWARD_Z'
            elif 'beta_over_h' in row and np.isfinite(row['beta_over_h']) and row['beta_over_h'] > 1.0:
                label = 'IMPULSE_EXPLODES_BETA_OVER_H'
            elif 'tangent_residual' in row and np.isfinite(row['tangent_residual']) and row['tangent_residual'] > 1.0e-2:
                label = 'CENTERING_DESTROYS_TANGENCY'
            else:
                label = 'NO_GAIN_OVER_FINAL_PROJECTION'
        rows_19.append({'test': '19_failure_labels', 'source_test': name, 'graph': row.get('graph'), 'failure_label': label, 'PASS': False, 'status': 'FAIL'})
df_19 = dataframe_result('19_failure_labels', rows_19)
display(df_19)

final_proj_df = df_16[df_16['method'] == 'KLDM_plus_final_fixed_projection'].copy() if not df_16.empty else pd.DataFrame()
rescue_df = df_16[df_16['method'] == 'impulse_only_refresh_z'].copy() if not df_16.empty else pd.DataFrame()
full_chain_beats = False
if not final_proj_df.empty and not rescue_df.empty:
    merged = rescue_df.merge(final_proj_df, on='graph', suffixes=('_rescue', '_baseline'))
    for _idx, row in merged.iterrows():
        if bool(row.get('valid_rescue', False) and row.get('requested_sg_match_rescue', False) and row.get('match_rescue', False)) and bool(row.get('valid_baseline', False) and row.get('requested_sg_match_baseline', False) and row.get('match_baseline', False)) and float(row.get('frac_rmse_rescue', float('inf'))) < float(row.get('frac_rmse_baseline', float('inf'))) - 1.0e-12:
            full_chain_beats = True
            break
summary_row = {
    'FIXED_TEMPLATE_VALID': bool((not df_1.empty) and df_1['PASS'].all()),
    'JACOBIAN_VALID': bool((not df_2.empty) and df_2['PASS'].all()),
    'SIGN_CONVENTION_VERIFIED': bool((not df_3.empty) and df_3['PASS'].all()),
    'IMPULSE_ALGEBRA_VALID': bool((not df_4.empty) and df_4['PASS'].all()),
    'IMPULSE_REDUCES_DISTANCE': bool((not df_5.empty) and df_5['PASS'].all()),
    'BETA_OVER_H_SAFE': bool((not df_6.empty) and np.isfinite(df_6['impulse_norm']).all()),
    'Z_UPDATE_STABLE': bool(not df_7.empty and np.isfinite(df_7['projection_energy']).all()),
    'DUAL_STABLE': bool(not df_8.empty and np.isfinite(df_8['mu_norm']).all()),
    'TANGENT_AFTER_MOVE_HELPFUL': bool((not df_9.empty) and ('impulse_move_then_tangent' in set(df_9.get('variant', [])))),
    'SCORE_SAFE': bool((not df_11.empty) and df_11['PASS'].all()),
    'MINI_CHAIN_STABLE': bool((not df_15.empty) and bool(df_15['PASS'].any())),
    'FULL_CHAIN_BEATS_FINAL_PROJECTION': bool(full_chain_beats),
    'CASAL_MIXING_TIME_GUARANTEE': False,
}
if summary_row['FULL_CHAIN_BEATS_FINAL_PROJECTION'] and summary_row['IMPULSE_REDUCES_DISTANCE']:
    recommendation = 'impulse_casal_rescue_branch'
elif summary_row['IMPULSE_REDUCES_DISTANCE']:
    recommendation = 'continue_to_chart_training'
else:
    recommendation = 'final_projection_only'
summary_row['RECOMMENDATION'] = recommendation
summary_row['CASAL_THEOREM_APPLIES'] = CASAL_THEOREM_APPLIES
summary_row['REASON'] = CASAL_THEOREM_REASON

df_20 = dataframe_result('20_final_summary_table', [summary_row])
display(df_20)


/workspace/.venv/lib/python3.11/site-packages/pymatgen/core/lattice.py:1109: RuntimeWarning: invalid value encountered in divide
  u[s - 1, : (s - 1)] = np.dot(a[:, s - 1].T, b[:, : (s - 1)]) / m[: (s - 1)]


 ** On entry to DLASCL parameter number  4 had an illegal value
 ** On entry to DLASCL parameter number  4 had an illegal value
 ** On entry to DLASCL parameter number  4 had an illegal value
 ** On entry to DLASCL parameter number  5 had an illegal value
 ** On entry to DLASCL parameter number  4 had an illegal value
 ** On entry to DLASCL parameter number  4 had an illegal value
 ** On entry to DLASCL parameter number  4 had an illegal value
 ** On entry to DLASCL parameter number  4 had an illegal value
 ** On entry to DLASCL parameter number  5 had an illegal value
 ** On entry to DLASCL parameter number  4 had an illegal value
 ** On entry to DLASCL parameter number  4 had an illegal value
 ** On entry to DLASCL parameter number  4 had an illegal value
 ** On entry to DLASCL parameter number  4 had an illegal value
 ** On entry to DLASCL parameter number  5 had an illegal value
 ** On entry to DLASCL parameter number  4 had an illegal value
 ** On entry to DLASCL parameter number 

,test,graph,tangent_mode,num_steps,dist_initial,dist_final,dist_monotonic_fraction,mu_norm_final,velocity_norm_max,PASS,status
0,12_synthetic_no_score_multi_step,2,none,8,1.770881,2.091660,0.428571,0.081733,1.898270,True,PASS
1,12_synthetic_no_score_multi_step,2,after_move_combined,8,1.770881,2.093111,0.428571,0.063406,0.654296,True,PASS
2,12_synthetic_no_score_multi_step,3,none,8,1.121658,0.010091,0.571429,0.008481,0.807870,True,PASS
3,12_synthetic_no_score_multi_step,3,after_move_combined,8,1.121658,0.000584,0.714286,0.009311,0.054776,True,PASS


,test,graph,variant,dist_before,dist_after_kldm_proposal,dist_after_variant,match,rmse,PASS,status
0,14_one_real_kldm_reverse_step,2,KLDM_baseline,1.893423,1.892751,1.892751,False,NaN,False,DIAG
1,14_one_real_kldm_reverse_step,2,final_projection_only,1.893423,1.892751,0.706539,False,NaN,False,DIAG
2,14_one_real_kldm_reverse_step,2,impulse_plus_cascal,1.893423,1.893283,1.848558,NaN,NaN,True,PASS
3,14_one_real_kldm_reverse_step,2,impulse_plus_cascal_plus_tangent,1.893423,1.893283,1.848558,NaN,NaN,True,PASS
4,14_one_real_kldm_reverse_step,3,KLDM_baseline,1.124981,1.125324,1.125324,False,NaN,False,DIAG
5,14_one_real_kldm_reverse_step,3,final_projection_only,1.124981,1.125324,0.003714,False,NaN,False,DIAG
6,14_one_real_kldm_reverse_step,3,impulse_plus_cascal,1.124981,1.125135,1.122056,NaN,NaN,True,PASS
7,14_one_real_kldm_reverse_step,3,impulse_plus_cascal_plus_tangent,1.124981,1.125135,1.122056,NaN,NaN,True,PASS


,test,graph,kappa_coord,beta,n_steps,dist_initial,dist_final,dist_best,velocity_norm_max,impulse_norm_max,...,projection_fail_count,match,rmse,frac_rmse,requested_sg_match,valid,min_pair_distance,volume_ratio,PASS,status
0,15_late_mini_chain,2,0.3,0.00375,12,1.708593,1.467871,1.467871,1.154359,0.318206,...,0,False,None,0.175514,False,False,0.384416,0.076748,False,DIAG
1,15_late_mini_chain,2,1.0,0.01250,12,1.707587,1.397584,1.397584,1.419494,1.060686,...,0,False,None,0.173081,False,False,0.254106,0.076748,False,DIAG
2,15_late_mini_chain,2,3.0,0.03750,12,1.701902,1.243276,1.243276,3.020020,3.182059,...,0,False,None,0.163798,False,False,0.126894,0.076748,False,DIAG
3,15_late_mini_chain,3,0.3,0.00375,12,1.342597,0.008551,0.008551,1.223069,0.163324,...,0,False,None,0.225021,None,False,0.000004,0.020002,False,DIAG
4,15_late_mini_chain,3,1.0,0.01250,12,1.339863,0.008557,0.008557,1.223068,0.544415,...,0,False,None,0.225020,None,False,0.000014,0.020002,False,DIAG
5,15_late_mini_chain,3,3.0,0.03750,12,1.332365,0.008575,0.008575,1.741704,1.633245,...,0,False,None,0.225018,None,False,0.000041,0.020002,False,DIAG


,test,graph,method,n_steps,dist_initial,dist_final,dist_best,velocity_norm_max,impulse_norm_max,score_norm_max,projection_fail_count,match,rmse,frac_rmse,requested_sg_match,valid,min_pair_distance,volume_ratio,PASS,status
0,16_full_reverse_chain_comparison,2,KLDM_plus_final_fixed_projection,40,1.731053,1.404448,1.388394,1.318599,7.340124e-07,321.007874,0,False,None,0.188020,False,False,6.184892e-02,1.841404e-03,False,FAIL
1,16_full_reverse_chain_comparison,2,impulse_only_refresh_z,40,1.730712,1.417843,1.388135,1.309352,7.340125e-02,314.334106,0,False,None,0.187541,False,False,1.368965e-01,1.841404e-03,False,FAIL
2,16_full_reverse_chain_comparison,2,impulse_casal_rescue_branch,40,1.730712,1.417843,1.388135,1.309352,7.340125e-02,296.074829,0,False,None,0.187541,False,False,1.368965e-01,1.841404e-03,False,FAIL
3,16_full_reverse_chain_comparison,2,impulse_casal_no_dual,40,1.730712,1.417843,1.388135,1.309352,7.340125e-02,296.074829,0,False,None,0.187541,False,False,1.368965e-01,1.841404e-03,False,FAIL
4,16_full_reverse_chain_comparison,3,KLDM_plus_final_fixed_projection,40,1.281700,0.011614,0.011614,1.014216,4.843571e-07,177.740753,0,False,None,0.225021,None,False,0.000000e+00,5.559902e-09,False,FAIL
5,16_full_reverse_chain_comparison,3,impulse_only_refresh_z,40,1.280521,0.011613,0.011613,1.014216,4.843571e-02,177.742096,0,False,None,0.225021,None,False,9.973960e-07,5.559902e-09,False,FAIL
6,16_full_reverse_chain_comparison,3,impulse_casal_rescue_branch,40,1.280521,0.011613,0.011613,1.014216,4.843571e-02,53.675674,0,False,None,0.225021,None,False,9.973960e-07,5.559902e-09,False,FAIL
7,16_full_reverse_chain_comparison,3,impulse_casal_no_dual,40,1.280521,0.011613,0.011613,1.014216,4.843571e-02,53.675674,0,False,None,0.225021,None,False,9.973960e-07,5.559902e-09,False,FAIL


,test,graph,theta_dim,rank_J,rank_ratio,PASS,status
0,17_dof_split,2,24,24,0.500000,True,PASS
1,17_dof_split,3,1,1,0.041667,True,PASS


,test,graph,beta,clip_norm,n_steps,dist_initial,dist_final,dist_best,velocity_norm_max,impulse_norm_max,...,projection_fail_count,match,rmse,frac_rmse,requested_sg_match,valid,min_pair_distance,volume_ratio,PASS,status
0,18_hyperparameter_robustness,2,0.0001,NaN,12,1.708408,1.500483,1.490326,1.223151,0.008485,...,0,False,None,0.177375,False,False,4.662896e-01,0.076748,False,DIAG
1,18_hyperparameter_robustness,2,0.0001,0.728792,12,1.708408,1.500483,1.490326,1.223151,0.008485,...,0,False,None,0.177375,False,False,4.662896e-01,0.076748,False,DIAG
2,18_hyperparameter_robustness,2,0.0001,2.186376,12,1.708408,1.500483,1.490326,1.223151,0.008485,...,0,False,None,0.177375,False,False,4.662896e-01,0.076748,False,DIAG
3,18_hyperparameter_robustness,2,0.0003,NaN,12,1.708418,1.498506,1.490401,1.217443,0.025456,...,0,False,None,0.177264,False,False,4.625928e-01,0.076748,False,DIAG
4,18_hyperparameter_robustness,2,0.0003,0.728792,12,1.708418,1.498506,1.490401,1.217443,0.025456,...,0,False,None,0.177264,False,False,4.625928e-01,0.076748,False,DIAG
5,18_hyperparameter_robustness,2,0.0003,2.186376,12,1.708418,1.498506,1.490401,1.217443,0.025456,...,0,False,None,0.177264,False,False,4.625928e-01,0.076748,False,DIAG
6,18_hyperparameter_robustness,2,0.0010,NaN,12,1.708452,1.491643,1.487844,1.199145,0.084855,...,0,False,None,0.176889,False,False,4.496808e-01,0.076748,False,DIAG
7,18_hyperparameter_robustness,2,0.0010,0.728792,12,1.708452,1.491643,1.487844,1.199145,0.084855,...,0,False,None,0.176889,False,False,4.496808e-01,0.076748,False,DIAG
8,18_hyperparameter_robustness,2,0.0010,2.186376,12,1.708452,1.491643,1.487844,1.199145,0.084855,...,0,False,None,0.176889,False,False,4.496808e-01,0.076748,False,DIAG
9,18_hyperparameter_robustness,2,0.0030,NaN,12,1.708552,1.474417,1.474417,1.162106,0.254565,...,0,False,None,0.175850,False,False,4.022509e-01,0.076748,False,DIAG


,test,source_test,graph,failure_label,PASS,status
0,19_failure_labels,7_cascal_z_update_correctness,2,NO_GAIN_OVER_FINAL_PROJECTION,False,FAIL
1,19_failure_labels,7_cascal_z_update_correctness,2,NO_GAIN_OVER_FINAL_PROJECTION,False,FAIL
2,19_failure_labels,7_cascal_z_update_correctness,2,NO_GAIN_OVER_FINAL_PROJECTION,False,FAIL
3,19_failure_labels,7_cascal_z_update_correctness,3,NO_GAIN_OVER_FINAL_PROJECTION,False,FAIL
4,19_failure_labels,7_cascal_z_update_correctness,3,NO_GAIN_OVER_FINAL_PROJECTION,False,FAIL
5,19_failure_labels,7_cascal_z_update_correctness,3,NO_GAIN_OVER_FINAL_PROJECTION,False,FAIL
6,19_failure_labels,11_one_step_kldm_score_ood,2,NO_GAIN_OVER_FINAL_PROJECTION,False,FAIL
7,19_failure_labels,14_one_real_kldm_reverse_step,2,NO_GAIN_OVER_FINAL_PROJECTION,False,FAIL
8,19_failure_labels,14_one_real_kldm_reverse_step,2,NO_GAIN_OVER_FINAL_PROJECTION,False,FAIL
9,19_failure_labels,14_one_real_kldm_reverse_step,3,NO_GAIN_OVER_FINAL_PROJECTION,False,FAIL


,FIXED_TEMPLATE_VALID,JACOBIAN_VALID,SIGN_CONVENTION_VERIFIED,IMPULSE_ALGEBRA_VALID,IMPULSE_REDUCES_DISTANCE,BETA_OVER_H_SAFE,Z_UPDATE_STABLE,DUAL_STABLE,TANGENT_AFTER_MOVE_HELPFUL,SCORE_SAFE,MINI_CHAIN_STABLE,FULL_CHAIN_BEATS_FINAL_PROJECTION,CASAL_MIXING_TIME_GUARANTEE,RECOMMENDATION,CASAL_THEOREM_APPLIES,REASON,PASS,status
0,True,True,True,True,True,True,True,True,True,False,False,False,False,continue_to_chart_training,False,kinetic_nonconvex_learned_score_wyckoff_union_...,False,FAIL
